In [2]:
#########################
# config + output paths #
#########################


from pathlib import Path

REPO_ROOT = Path("/Users/mohamadyassin/Documents/AIRC/home_page")  # change if needed
OD = REPO_ROOT / "open-data"
REG = OD / "registry"
STAGING = OD / "staging" / "raw"

REG.mkdir(parents=True, exist_ok=True)
STAGING.mkdir(parents=True, exist_ok=True)

print("OK:", REG)

OK: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry


In [4]:
# ============================================================
# CELL B — Pull opengeos TSV lists → write raw registry Parquet
# (robust: forces all columns to STRING, handles mixed schemas)
# ============================================================

import pandas as pd

OPEN_GEOS_BASE = "https://raw.githubusercontent.com/opengeos/geospatial-data-catalogs/master"

TSV_SOURCES = [
    ("aws_open_datasets", f"{OPEN_GEOS_BASE}/aws_open_datasets.tsv"),
    ("aws_geo_datasets",  f"{OPEN_GEOS_BASE}/aws_geo_datasets.tsv"),
    ("aws_stac_catalogs", f"{OPEN_GEOS_BASE}/aws_stac_catalogs.tsv"),
    ("stac_catalogs",     f"{OPEN_GEOS_BASE}/stac_catalogs.tsv"),
    ("gee_catalog",       f"{OPEN_GEOS_BASE}/gee_catalog.tsv"),
    ("pc_catalog",        f"{OPEN_GEOS_BASE}/pc_catalog.tsv"),
    ("nasa_cmr_catalog",  f"{OPEN_GEOS_BASE}/nasa_cmr_catalog.tsv"),
]

dfs = []
all_cols = set()

for name, url in TSV_SOURCES:
    # Read as strings only (prevents ArrowInvalid due to mixed types)
    df = pd.read_csv(
        url,
        sep="\t",
        dtype="string",           # <-- key: no mixed numeric/object inference
        keep_default_na=False,    # keep empty strings as empty, not NaN
        na_values=[],             # don't invent NaNs
        encoding_errors="replace"
    )

    # Add provenance
    df["source_name"] = name
    df["source_url"] = url

    # Normalize: strip whitespace in column names and values
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        # ensure pandas StringDtype + strip values
        df[c] = df[c].astype("string").str.strip()

    dfs.append(df)
    all_cols |= set(df.columns)

    print(f"{name}: shape={df.shape} cols={len(df.columns)}")

# Union columns across all sources
all_cols = sorted(all_cols)
dfs2 = []
for df in dfs:
    missing = [c for c in all_cols if c not in df.columns]
    for c in missing:
        df[c] = pd.Series([pd.NA] * len(df), dtype="string")
    dfs2.append(df[all_cols])

raw = pd.concat(dfs2, ignore_index=True)

# Final safety: force ALL to string dtype
for c in raw.columns:
    raw[c] = raw[c].astype("string")

out_path = REG / "datasets_opengeos_raw.parquet"
raw.to_parquet(out_path, index=False)

print("WROTE:", out_path)
print("RAW SHAPE:", raw.shape)

raw.head(5)

aws_open_datasets: shape=(1635, 18) cols=18
aws_geo_datasets: shape=(940, 18) cols=18
aws_stac_catalogs: shape=(90, 9) cols=9
stac_catalogs: shape=(112, 13) cols=13
gee_catalog: shape=(835, 16) cols=16
pc_catalog: shape=(126, 18) cols=18
nasa_cmr_catalog: shape=(22330, 11) cols=11
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/datasets_opengeos_raw.parquet
RAW SHAPE: (26068, 51)


,ARN,AccountRequired,Contact,ControlledAccess,Description,Documentation,Endpoint,Explore,Host,License,...,source_url,start_date,state_date,storage_account,summary,title,type,updated,url,variables
0,arn:aws:s3:::noaa-nws-fourcastnetgfs-pds,,For questions regarding data content or qualit...,,FourCastNet GFS data,"For the NOAA Product, https://fourcastnetgfs.r...",<NA>,['[Browse Bucket](https://noaa-nws-fourcastnet...,,NOAA's FourCastNetGFS products are released un...,...,https://raw.githubusercontent.com/opengeos/geo...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,arn:aws:sns:us-east-1:709902155096:NewNWSFOURC...,,For questions regarding data content or qualit...,,"New data notifications for FourCastNet GFS, on...","For the NOAA Product, https://fourcastnetgfs.r...",<NA>,,,NOAA's FourCastNetGFS products are released un...,...,https://raw.githubusercontent.com/opengeos/geo...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,arn:aws:s3:::1000genomes,,http://www.internationalgenome.org/contact,,http://wwwinternationalgenomeorg/formats,https://github.com/awslabs/open-data-docs/tree...,<NA>,,,Data from the 1000 Genomes Project is now avai...,...,https://raw.githubusercontent.com/opengeos/geo...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,arn:aws:s3:::1000genomes-dragen,,"[Illumina, Inc.](mailto:support@illumina.com)",,"BAM, SNV-vcf, SNV-gvcf, STR-vcf, STR-bam, SV-v...",[DRAGEN Support Resources](https://support.ill...,<NA>,,,TBD,...,https://raw.githubusercontent.com/opengeos/geo...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,arn:aws:s3:::1000genomes-dragen-3.7.6,,"[Illumina, Inc.](mailto:support@illumina.com)",,"BAM, SNV-vcf, SNV-gvcf, STR-vcf, STR-bam, SV-v...",[DRAGEN Support Resources](https://support.ill...,<NA>,,,TBD,...,https://raw.githubusercontent.com/opengeos/geo...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [13]:
# ============================================================
# CELL C — Build AIRC canonical registry (per-source parsers)
# (fixed: pd.NA-safe IDs + strict schema)
# ============================================================

import pandas as pd
import hashlib

raw = pd.read_parquet(REG / "datasets_opengeos_raw.parquet").copy()

CANON_COLS = [
    "dataset_id",
    "title",
    "summary",
    "data_kind",
    "access_kind",
    "homepage_url",
    "endpoint_url",
    "provider",
    "license",
    "tags",
    "spatial_extent",
    "temporal_extent",
    "source_name",
    "source_url",
]

def stable_id(*parts) -> str:
    s = "|".join([(p if p is not None else "") for p in parts])
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def safe_str(x) -> str:
    # pd.NA / NaN safe
    if x is None:
        return ""
    if pd.isna(x):
        return ""
    return str(x)

def mk_df(
    df,
    *,
    source_name,
    source_url,
    title=None,
    summary=None,
    data_kind="unknown",
    access_kind="unknown",
    homepage_url=None,
    endpoint_url=None,
    provider=None,
    license=None,
    tags=None,
    spatial_extent=None,
    temporal_extent=None,
):
    out = pd.DataFrame({
        "title": title,
        "summary": summary,
        "data_kind": data_kind,
        "access_kind": access_kind,
        "homepage_url": homepage_url,
        "endpoint_url": endpoint_url,
        "provider": provider,
        "license": license,
        "tags": tags,
        "spatial_extent": spatial_extent,
        "temporal_extent": temporal_extent,
    })

    out["source_name"] = source_name
    out["source_url"] = source_url

    # build stable IDs (pd.NA-safe)
    out["dataset_id"] = [
        stable_id(source_name, safe_str(t), safe_str(h), safe_str(e))
        for t, h, e in zip(out["title"], out["homepage_url"], out["endpoint_url"])
    ]

    # enforce column order + string dtypes
    out = out[CANON_COLS]
    for c in CANON_COLS:
        out[c] = out[c].astype("string")

    return out

out_parts = []

# ---- 1) AWS open datasets + AWS geo datasets (same schema) ----
for src in ["aws_open_datasets", "aws_geo_datasets"]:
    df = raw[raw["source_name"] == src].copy()
    out_parts.append(mk_df(
        df,
        source_name=src,
        source_url=df["source_url"].iloc[0],
        title=df.get("Name"),
        summary=df.get("Description"),
        data_kind="unknown",
        access_kind="aws_open_data",
        homepage_url=df.get("Documentation"),
        endpoint_url=df.get("ARN"),
        provider=df.get("ManagedBy"),
        license=df.get("License"),
        tags=df.get("Tags"),
        spatial_extent=pd.NA,
        temporal_extent=df.get("UpdateFrequency"),
    ))

# ---- 2) AWS STAC catalogs ----
src = "aws_stac_catalogs"
df = raw[raw["source_name"] == src].copy()

title_col = "name" if "name" in df.columns else ("title" if "title" in df.columns else None)
url_col   = "url"  if "url"  in df.columns else ("stac_url" if "stac_url" in df.columns else None)

out_parts.append(mk_df(
    df,
    source_name=src,
    source_url=df["source_url"].iloc[0],
    title=df[title_col] if title_col else pd.NA,
    summary=pd.NA,
    data_kind="unknown",
    access_kind="stac_catalog",
    homepage_url=pd.NA,
    endpoint_url=df[url_col] if url_col else pd.NA,
    provider=pd.NA,
    license=pd.NA,
    tags=pd.NA,
    spatial_extent=pd.NA,
    temporal_extent=pd.NA,
))

# ---- 3) STAC catalogs list (STAC Index catalogs via opengeos) ----
src = "stac_catalogs"
df = raw[raw["source_name"] == src].copy()

title = df["name"] if "name" in df.columns else (df["title"] if "title" in df.columns else pd.NA)
endpoint = df["url"] if "url" in df.columns else pd.NA
provider = df["provider"] if "provider" in df.columns else pd.NA

out_parts.append(mk_df(
    df,
    source_name=src,
    source_url=df["source_url"].iloc[0],
    title=title,
    summary=df["description"] if "description" in df.columns else pd.NA,
    data_kind="unknown",
    access_kind="stac_catalog",
    homepage_url=pd.NA,
    endpoint_url=endpoint,
    provider=provider,
    license=df["license"] if "license" in df.columns else pd.NA,
    tags=df["keywords"] if "keywords" in df.columns else pd.NA,
    spatial_extent=pd.NA,
    temporal_extent=pd.NA,
))

# ---- 4) Planetary Computer catalog list ----
src = "pc_catalog"
df = raw[raw["source_name"] == src].copy()

title = df["name"] if "name" in df.columns else (df["title"] if "title" in df.columns else pd.NA)
endpoint = df["stac_url"] if "stac_url" in df.columns else (df["url"] if "url" in df.columns else pd.NA)

out_parts.append(mk_df(
    df,
    source_name=src,
    source_url=df["source_url"].iloc[0],
    title=title,
    summary=df["description"] if "description" in df.columns else pd.NA,
    data_kind="unknown",
    access_kind="stac_catalog",
    homepage_url=df["url"] if "url" in df.columns else pd.NA,
    endpoint_url=endpoint,
    provider="Microsoft Planetary Computer",
    license=df["license"] if "license" in df.columns else pd.NA,
    tags=df["keywords"] if "keywords" in df.columns else pd.NA,
    spatial_extent=pd.NA,
    temporal_extent=pd.NA,
))

# ---- 5) Google Earth Engine catalog list ----
src = "gee_catalog"
df = raw[raw["source_name"] == src].copy()

title = df["name"] if "name" in df.columns else (df["title"] if "title" in df.columns else pd.NA)

out_parts.append(mk_df(
    df,
    source_name=src,
    source_url=df["source_url"].iloc[0],
    title=title,
    summary=df["description"] if "description" in df.columns else pd.NA,
    data_kind="raster",
    access_kind="gee",
    homepage_url=df["url"] if "url" in df.columns else pd.NA,
    endpoint_url=pd.NA,
    provider="Google Earth Engine",
    license=df["license"] if "license" in df.columns else pd.NA,
    tags=df["tags"] if "tags" in df.columns else pd.NA,
    spatial_extent=pd.NA,
    temporal_extent=pd.NA,
))

# ---- 6) NASA CMR catalog list (huge) — minimal for now ----
src = "nasa_cmr_catalog"
df = raw[raw["source_name"] == src].copy()

title = df["title"] if "title" in df.columns else pd.NA
homepage = df["url"] if "url" in df.columns else pd.NA

out_parts.append(mk_df(
    df,
    source_name=src,
    source_url=df["source_url"].iloc[0],
    title=title,
    summary=df["summary"] if "summary" in df.columns else pd.NA,
    data_kind="unknown",
    access_kind="cmr",
    homepage_url=homepage,
    endpoint_url=pd.NA,
    provider="NASA CMR",
    license=df["license"] if "license" in df.columns else pd.NA,
    tags=df["keywords"] if "keywords" in df.columns else pd.NA,
    spatial_extent=pd.NA,
    temporal_extent=pd.NA,
))

datasets = pd.concat(out_parts, ignore_index=True)

# Deduplicate by dataset_id
datasets = datasets.drop_duplicates(subset=["dataset_id"]).reset_index(drop=True)

datasets_path = REG / "datasets.parquet"
datasets.to_parquet(datasets_path, index=False)

sources = (
    datasets[["source_name", "source_url"]]
    .drop_duplicates()
    .copy()
)
sources["source_kind"] = "opengeos_tsv"
sources["notes"] = ""
sources = sources[["source_name", "source_kind", "source_url", "notes"]]
for c in sources.columns:
    sources[c] = sources[c].astype("string")

sources_path = REG / "sources.parquet"
sources.to_parquet(sources_path, index=False)

print("WROTE:", datasets_path, "rows=", len(datasets))
print("WROTE:", sources_path,  "rows=", len(sources))

print("\nDataset counts by access_kind:")
print(datasets["access_kind"].value_counts().head(20))

print("\nSample rows (stac_catalog):")
display(datasets[datasets["access_kind"]=="stac_catalog"][["title","endpoint_url","provider","source_name"]].head(10))

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/datasets.parquet rows= 24189
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/sources.parquet rows= 7

Dataset counts by access_kind:
access_kind
cmr              20592
aws_open_data     2523
gee                835
stac_catalog       239
Name: count, dtype: Int64

Sample rows (stac_catalog):


,title,endpoint_url,provider,source_name
2523,<NA>,<NA>,<NA>,aws_stac_catalogs
2524,African Agriculture Adaptation Atlas STAC catalog,https://digital-atlas.s3.amazonaws.com/stac/pu...,<NA>,stac_catalogs
2525,Astraea Earth OnDemand,https://eod-catalog-svc-prod.astraea.earth/,<NA>,stac_catalogs
2526,BON in a Box STAC,https://stac.geobon.org/,<NA>,stac_catalogs
2527,CBERS and Amazonia-1 on AWS,https://stac.scitekno.com.br/v100/,<NA>,stac_catalogs
2528,CIESIN STAC,https://ciesin.github.io/sci-apps-stac/stac/ca...,<NA>,stac_catalogs
2529,CREODIAS,https://datahub.creodias.eu/stac/,<NA>,stac_catalogs
2530,California Forest Observatory,https://storage.googleapis.com/cfo-public/cata...,<NA>,stac_catalogs
2531,Canadian Geospatial Data Collections,https://datacube.services.geo.ca/stac/api/,<NA>,stac_catalogs
2532,Capella Space Open Data,https://capella-open-data.s3.us-west-2.amazona...,<NA>,stac_catalogs


In [14]:
import pandas as pd
datasets = pd.read_parquet(REG / "datasets.parquet")
datasets[["access_kind","title","homepage_url","endpoint_url","provider","source_name"]].head(20)

,access_kind,title,homepage_url,endpoint_url,provider,source_name
0,aws_open_data,(EXPERIMENTAL) NOAA FourCastNet Global Forecas...,"For the NOAA Product, https://fourcastnetgfs.r...",arn:aws:s3:::noaa-nws-fourcastnetgfs-pds,[NOAA](http://www.noaa.gov/),aws_open_datasets
1,aws_open_data,(EXPERIMENTAL) NOAA FourCastNet Global Forecas...,"For the NOAA Product, https://fourcastnetgfs.r...",arn:aws:sns:us-east-1:709902155096:NewNWSFOURC...,[NOAA](http://www.noaa.gov/),aws_open_datasets
2,aws_open_data,1000 Genomes,https://github.com/awslabs/open-data-docs/tree...,arn:aws:s3:::1000genomes,National Institutes of Health,aws_open_datasets
3,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,[DRAGEN Support Resources](https://support.ill...,arn:aws:s3:::1000genomes-dragen,"[Illumina, Inc.](https://www.illumina.com/prod...",aws_open_datasets
4,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,[DRAGEN Support Resources](https://support.ill...,arn:aws:s3:::1000genomes-dragen-3.7.6,"[Illumina, Inc.](https://www.illumina.com/prod...",aws_open_datasets
5,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,[DRAGEN Support Resources](https://support.ill...,arn:aws:s3:::1000genomes-dragen-v3.7.6,"[Illumina, Inc.](https://www.illumina.com/prod...",aws_open_datasets
6,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,[DRAGEN Support Resources](https://support.ill...,arn:aws:s3:::1000genomes-dragen-v4.0.3,"[Illumina, Inc.](https://www.illumina.com/prod...",aws_open_datasets
7,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,[DRAGEN Support Resources](https://support.ill...,arn:aws:s3:::1000genomes-dragen-v4-2-7,"[Illumina, Inc.](https://www.illumina.com/prod...",aws_open_datasets
8,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,[DRAGEN Support Resources](https://support.ill...,arn:aws:s3:::1000genomes-dragen-v4-4-7,"[Illumina, Inc.](https://www.illumina.com/prod...",aws_open_datasets
9,aws_open_data,10m Annual Land Use Land Cover (9-class),https://www.impactobservatory.com/global_maps,arn:aws:s3:::io-10m-annual-lulc,[Impact Observatory](https://www.impactobserva...,aws_open_datasets


In [15]:
import pandas as pd

datasets = pd.read_parquet(REG / "datasets.parquet")
sources  = pd.read_parquet(REG / "sources.parquet")

print("datasets:", datasets.shape)
print("sources:", sources.shape)

print("\nUnique dataset_id:", datasets["dataset_id"].nunique())
print("Duplicate dataset_id rows:", datasets.shape[0] - datasets["dataset_id"].nunique())

print("\nCounts by access_kind:")
print(datasets["access_kind"].value_counts(dropna=False))

print("\nCounts by source_name:")
print(datasets["source_name"].value_counts(dropna=False))

datasets: (24189, 14)
sources: (7, 4)

Unique dataset_id: 24189
Duplicate dataset_id rows: 0

Counts by access_kind:
access_kind
cmr              20592
aws_open_data     2523
gee                835
stac_catalog       239
Name: count, dtype: Int64

Counts by source_name:
source_name
nasa_cmr_catalog     20592
aws_open_datasets     1583
aws_geo_datasets       940
gee_catalog            835
pc_catalog             126
stac_catalogs          112
aws_stac_catalogs        1
Name: count, dtype: Int64


In [16]:
import pandas as pd

datasets = pd.read_parquet(REG / "datasets.parquet")

fields = ["title","summary","homepage_url","endpoint_url","provider","license","tags"]
coverage = (datasets[fields].notna().mean().sort_values(ascending=False) * 100).round(2)

print("Field coverage %:")
display(coverage.to_frame("percent_non_null"))

print("\nCoverage by access_kind (percent_non_null):")
by_kind = datasets.groupby("access_kind")[fields].apply(lambda g: (g.notna().mean()*100).round(2))
display(by_kind)

Field coverage %:


,percent_non_null
title,100.00
provider,99.53
license,99.53
homepage_url,99.01
summary,10.95
tags,10.95
endpoint_url,10.89



Coverage by access_kind (percent_non_null):


,title,summary,homepage_url,endpoint_url,provider,license,tags
access_kind,,,,,,,
aws_open_data,100.00,100.00,100.0,100.00,100.00,100.00,100.00
cmr,100.00,0.00,100.0,0.00,100.00,100.00,0.00
gee,100.00,0.00,100.0,0.00,100.00,100.00,0.00
stac_catalog,99.58,52.72,0.0,46.86,52.72,52.72,52.72


In [17]:
import pandas as pd
import re

datasets = pd.read_parquet(REG / "datasets.parquet")

url_re = re.compile(r"https?://", re.IGNORECASE)

def has_http(s):
    if s is None or pd.isna(s):
        return False
    return bool(url_re.search(str(s)))

datasets["homepage_has_http"] = datasets["homepage_url"].apply(has_http)
datasets["endpoint_has_http"] = datasets["endpoint_url"].apply(has_http)

print("homepage_url contains http %:", round(datasets["homepage_has_http"].mean()*100, 2))
print("endpoint_url contains http %:", round(datasets["endpoint_has_http"].mean()*100, 2))

print("\nEndpoints that are NOT http (sample 20):")
display(
    datasets[~datasets["endpoint_has_http"] & datasets["endpoint_url"].notna()]
    [["access_kind","title","endpoint_url","source_name"]]
    .head(20)
)

print("\nHomepages that are NOT http (sample 20):")
display(
    datasets[~datasets["homepage_has_http"] & datasets["homepage_url"].notna()]
    [["access_kind","title","homepage_url","source_name"]]
    .head(20)
)

homepage_url contains http %: 98.8
endpoint_url contains http %: 0.46

Endpoints that are NOT http (sample 20):


,access_kind,title,endpoint_url,source_name
0,aws_open_data,(EXPERIMENTAL) NOAA FourCastNet Global Forecas...,arn:aws:s3:::noaa-nws-fourcastnetgfs-pds,aws_open_datasets
1,aws_open_data,(EXPERIMENTAL) NOAA FourCastNet Global Forecas...,arn:aws:sns:us-east-1:709902155096:NewNWSFOURC...,aws_open_datasets
2,aws_open_data,1000 Genomes,arn:aws:s3:::1000genomes,aws_open_datasets
3,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,arn:aws:s3:::1000genomes-dragen,aws_open_datasets
4,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,arn:aws:s3:::1000genomes-dragen-3.7.6,aws_open_datasets
5,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,arn:aws:s3:::1000genomes-dragen-v3.7.6,aws_open_datasets
6,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,arn:aws:s3:::1000genomes-dragen-v4.0.3,aws_open_datasets
7,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,arn:aws:s3:::1000genomes-dragen-v4-2-7,aws_open_datasets
8,aws_open_data,1000 Genomes Phase 3 Reanalysis with DRAGEN 3....,arn:aws:s3:::1000genomes-dragen-v4-4-7,aws_open_datasets
9,aws_open_data,10m Annual Land Use Land Cover (9-class),arn:aws:s3:::io-10m-annual-lulc,aws_open_datasets



Homepages that are NOT http (sample 20):


,access_kind,title,homepage_url,source_name
13,aws_open_data,2010 Census Production Settings Demographic an...,[2010 Census Production Settings Demographic a...,aws_open_datasets
14,aws_open_data,2010 Census Production Settings Demographic an...,[2010 Census Production Settings Demographic a...,aws_open_datasets
15,aws_open_data,2010 Census Production Settings Redistricting ...,[2010 Census Production Settings Redistricting...,aws_open_datasets
16,aws_open_data,2010 Census Production Settings Redistricting ...,[2010 Census Production Settings Redistricting...,aws_open_datasets
17,aws_open_data,2020 Census Demographic and Housing Characteri...,[Demographic and Housing Characteristics (DHC)...,aws_open_datasets
18,aws_open_data,2020 Census Demographic and Housing Characteri...,[Demographic and Housing Characteristics (DHC)...,aws_open_datasets
19,aws_open_data,2020 Census Redistricting Data (P.L. 94-171) N...,[2020 Census Redistricting Data (P.L. 94-171) ...,aws_open_datasets
20,aws_open_data,2020 Census Redistricting Data (P.L. 94-171) N...,[2020 Census Redistricting Data (P.L. 94-171) ...,aws_open_datasets
110,aws_open_data,Aurora Multi-Sensor Dataset,A third-party development kit authored by Andr...,aws_open_datasets
187,aws_open_data,CRC-SAS/SISSA historical seasonal and subseaso...,"General information, tutorials and examples, c...",aws_open_datasets


In [18]:
import pandas as pd

datasets = pd.read_parquet(REG / "datasets.parquet")

aws = datasets[datasets["access_kind"]=="aws_open_data"].copy()

txt = (
    aws["title"].fillna("") + " " +
    aws["summary"].fillna("") + " " +
    aws["tags"].fillna("")
).str.lower()

geo_kw = ["gis","geospatial","satellite","landsat","sentinel","lidar","dem","elevation","imagery","raster","vector","osm","road","hydro","noaa","usgs","census","boundaries"]
aws["geo_score"] = sum(txt.str.contains(k) for k in geo_kw)

print("AWS open data rows:", len(aws))
print("Rows with geo_score >= 1:", (aws["geo_score"]>=1).sum())
print("Rows with geo_score >= 2:", (aws["geo_score"]>=2).sum())

display(aws.sort_values("geo_score", ascending=False)[["title","geo_score","tags","homepage_url","endpoint_url"]].head(25))

AWS open data rows: 2523
Rows with geo_score >= 1: 1719
Rows with geo_score >= 2: 974


,title,geo_score,tags,homepage_url,endpoint_url
250,Copernicus Digital Elevation Model (DEM),5,"aws-pds, agriculture, elevation, earth observa...",https://copernicus-dem-30m.s3.amazonaws.com/re...,arn:aws:s3:::copernicus-dem-90m
2499,"USGS Landsat - New scene notifications, Level-...",5,"aws-pds, agriculture, earth observation, satel...",https://www.usgs.gov/core-science-systems/nli/...,arn:aws:sns:us-west-2:673253540267:public-c2-n...
2277,NUVIEW - Multi-State Geospatial Data - Imagery,5,"aws-pds, geospatial, satellite imagery, natura...",Documentation is available for this data at th...,arn:aws:s3:::nuview-state-opendata
2278,NUVIEW - Multi-State Geospatial Data - New dat...,5,"aws-pds, geospatial, satellite imagery, natura...",Documentation is available for this data at th...,arn:aws:sns:us-west-2:830737158982:nuview-stat...
1082,NUVIEW - Multi-State Geospatial Data,5,"aws-pds, geospatial, satellite imagery, natura...",Documentation is available for this data at th...,arn:aws:sns:us-west-2:830737158982:nuview-stat...
1081,NUVIEW - Multi-State Geospatial Data,5,"aws-pds, geospatial, satellite imagery, natura...",Documentation is available for this data at th...,arn:aws:s3:::nuview-state-opendata
434,EarthDEM,5,"aws-pds, elevation, earth observation, geospat...",https://www.pgc.umn.edu/data/earthdem/,arn:aws:s3:::pgc-opendata-dems/earthdem/strips/
1683,Copernicus Digital Elevation Model (DEM) - GLO...,5,"aws-pds, agriculture, elevation, earth observa...",https://copernicus-dem-30m.s3.amazonaws.com/re...,arn:aws:s3:::copernicus-dem-30m
2501,USGS Landsat - Scenes and metadata,5,"aws-pds, agriculture, earth observation, satel...",https://www.usgs.gov/core-science-systems/nli/...,arn:aws:s3:::usgs-landsat/collection02/
2500,"USGS Landsat - New scene notifications, US ARD...",5,"aws-pds, agriculture, earth observation, satel...",https://www.usgs.gov/core-science-systems/nli/...,arn:aws:sns:us-west-2:673253540267:public-c2-a...


In [19]:
import pandas as pd

datasets = pd.read_parquet(REG / "datasets.parquet").copy()

def endpoint_kind(x):
    if x is None or pd.isna(x):
        return "none"
    s = str(x)
    if s.startswith("arn:aws:s3:::"):
        return "aws_s3_arn"
    if s.startswith("arn:aws:sns:"):
        return "aws_sns_arn"
    if s.startswith("http://") or s.startswith("https://"):
        return "http_url"
    return "other"

datasets["endpoint_kind"] = datasets["endpoint_url"].apply(endpoint_kind)

print(datasets["endpoint_kind"].value_counts(dropna=False))
print("\nBy access_kind:")
print(pd.crosstab(datasets["access_kind"], datasets["endpoint_kind"]))

endpoint_kind
none           21554
aws_s3_arn      2119
aws_sns_arn      386
http_url         112
other             18
Name: count, dtype: int64

By access_kind:
endpoint_kind  aws_s3_arn  aws_sns_arn  http_url   none  other
access_kind                                                   
aws_open_data        2119          386         0      0     18
cmr                     0            0         0  20592      0
gee                     0            0         0    835      0
stac_catalog            0            0       112    127      0


In [22]:
# ============================================================
# CELL D — ArcGIS Hub (SanGIS) via /api/v3/datasets
# Robust: stops on server-side pagination errors (500), max_pages cap
# ============================================================

import pandas as pd
import requests

SANGIS_HOST = "https://gis-sangis1.hub.arcgis.com"
DATASETS_URL = f"{SANGIS_HOST}/api/v3/datasets"

FIELDS = "name,description,url,tags,license,modified,created,owner,orgId,access,source"

def fetch_page(page_number: int, page_size: int = 100):
    params = {
        "fields[datasets]": FIELDS,
        "sort": "-modified",
        "page[number]": page_number,
        "page[size]": page_size,
    }
    r = requests.get(DATASETS_URL, params=params, timeout=60)

    # If Hub errors on deep pages, treat as end-of-results
    if r.status_code >= 500:
        return {"_error": f"{r.status_code} server error", "data": [], "meta": {}, "links": {}}

    r.raise_for_status()
    return r.json()

rows = []
page = 1
page_size = 100
max_pages = 200   # safety cap

while page <= max_pages:
    js = fetch_page(page, page_size=page_size)

    if "_error" in js:
        print(f"STOP: server error at page {page} -> treating as end.")
        break

    data = js.get("data", []) or []
    if not data:
        print(f"STOP: empty page at {page}")
        break

    for rec in data:
        attrs = rec.get("attributes", {}) or {}
        links = rec.get("links", {}) or {}

        rows.append({
            "title": attrs.get("name"),
            "summary": attrs.get("description"),
            "data_kind": "vector",
            "access_kind": "arcgis_hub",
            "homepage_url": links.get("itemPage") or links.get("self"),
            "endpoint_url": links.get("esriRest") or attrs.get("url") or links.get("self"),
            "provider": attrs.get("owner") or attrs.get("orgId"),
            "license": attrs.get("license"),
            "tags": ",".join(attrs.get("tags", [])) if isinstance(attrs.get("tags"), list) else attrs.get("tags"),
            "spatial_extent": pd.NA,
            "temporal_extent": pd.NA,
            "source_name": "arcgis_hub_sangis",
            "source_url": SANGIS_HOST,

            # extra trace
            "hub_dataset_id": rec.get("id"),
            "modified": attrs.get("modified"),
        })

    # stop condition: fewer than page_size means last page
    if len(data) < page_size:
        print(f"STOP: last page {page} (returned {len(data)} < {page_size})")
        break

    page += 1

arcgis_datasets = pd.DataFrame(rows)

arcgis_datasets.to_parquet(REG / "arcgis_datasets.parquet", index=False)

arcgis_hubs = pd.DataFrame([{
    "source_name": "arcgis_hub_sangis",
    "source_kind": "arcgis_hub",
    "source_url": SANGIS_HOST,
    "notes": "SanGIS Hub seed (datasets via /api/v3/datasets)"
}])
arcgis_hubs.to_parquet(REG / "arcgis_hubs.parquet", index=False)

print("WROTE:", REG / "arcgis_datasets.parquet", "rows=", len(arcgis_datasets))
print("WROTE:", REG / "arcgis_hubs.parquet", "rows=", len(arcgis_hubs))

arcgis_datasets[["title","homepage_url","endpoint_url","license","tags","hub_dataset_id"]].head(15)

STOP: server error at page 101 -> treating as end.
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/arcgis_datasets.parquet rows= 10000
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/arcgis_hubs.parquet rows= 1


,title,homepage_url,endpoint_url,license,tags,hub_dataset_id
0,NSW Imagery Web Service GDA94,https://www.arcgis.com/home/item.html?id=ca937...,https://www.arcgis.com/sharing/content/items/c...,custom,"ADS,ADS Sensor,Aerial Imagery Mosa,Aerial Imag...",ca93748ebe5a46f1ae5451f35b3d0b9d_0
1,Development Plan 2023 2029 RPS Record of Prote...,https://www.arcgis.com/home/item.html?id=60138...,https://www.arcgis.com/sharing/content/items/6...,CC-BY-4.0,"DP2329,Planning & Cadastral,Planning,Planning ...",601387c1d5ed4f6aa172fcf231a61fd3_0
2,Development Plan 2023 2029 Specific Objectives...,https://www.arcgis.com/home/item.html?id=4dc2e...,https://www.arcgis.com/sharing/content/items/4...,CC-BY-4.0,"DP2329,Planning & Cadastral,Planning,Planning ...",4dc2eb5165c24ee2963a05c85c53cb6c_0
3,Development Plan 2023 2029 ACA Architectural C...,https://www.arcgis.com/home/item.html?id=5c7e4...,https://www.arcgis.com/sharing/content/items/5...,CC-BY-4.0,"Planning & Cadastral,Planning,Planning and Lan...",5c7e4fd86fd5485192b2a5ed059561e2_0
4,Development Plan 2023 2029 LAP Local Area Plan...,https://www.arcgis.com/home/item.html?id=d7416...,https://www.arcgis.com/sharing/content/items/d...,CC-BY-4.0,"DP2329,Planning,Planning & Cadastral,Planning ...",d7416617c8564962828191f1a66ce097_0
5,DAS Batang toru,https://www.arcgis.com/home/item.html?id=117de...,https://www.arcgis.com/sharing/content/items/1...,none,"DAS,Sibolga,Sumatra",117de73830ff4b63a5098e6aafe4de49_0
6,Dublin Airport Noise Zones Development Plan 20...,https://www.arcgis.com/home/item.html?id=499ad...,https://www.arcgis.com/sharing/content/items/4...,custom,"Community Information,Development Plan 2023-20...",499adb2858fe4678afc4eaa66540d279_0
7,Cycle Lanes Protected Rivervalley/Hartstown FCC,https://www.arcgis.com/home/item.html?id=a8c57...,https://www.arcgis.com/sharing/content/items/a...,custom,"Transportation,Transport,Active Travel,Roads a...",a8c57643d0f745ef97e6eaff80e22b72_0
8,Protected Greenway Cycling Lanes FCC,https://www.arcgis.com/home/item.html?id=68dc0...,https://www.arcgis.com/sharing/content/items/6...,custom,"Transport,Active Travel,Cycling,Transportation...",68dc0568eaf54f3b9eb7e5cf6883b879_0
9,Reserveringsgebieden waterberging,https://www.arcgis.com/home/item.html?id=65998...,https://www.arcgis.com/sharing/content/items/6...,custom,"Kunstwerken,Watersysteem,Waterbeheerplan 5,Bel...",659980984708489faaff0afeb2ab1f23_0


In [25]:
# ============================================================
# CELL D — ArcGIS Hub GLOBAL dataset registry (10k cap safe)
# ============================================================

import pandas as pd
import requests

HOST = "https://hub.arcgis.com"
DATASETS_URL = f"{HOST}/api/v3/datasets"

FIELDS = "name,description,url,tags,license,modified,created,owner,orgId,access,source"

def fetch_page(page_number: int, page_size: int = 100):
    params = {
        "fields[datasets]": FIELDS,
        "sort": "-modified",
        "page[number]": page_number,
        "page[size]": page_size,
    }
    r = requests.get(DATASETS_URL, params=params, timeout=60)
    if r.status_code >= 500:
        return {"_error": f"{r.status_code} server error", "data": []}
    r.raise_for_status()
    return r.json()

rows = []
page = 1
page_size = 100
max_pages = 100   # 100 pages * 100 = 10,000 (avoid the known deep-page crash)

while page <= max_pages:
    js = fetch_page(page, page_size=page_size)

    if "_error" in js:
        print(f"STOP: server error at page {page}")
        break

    data = js.get("data", []) or []
    if not data:
        print(f"STOP: empty page at {page}")
        break

    for rec in data:
        attrs = rec.get("attributes", {}) or {}
        links = rec.get("links", {}) or {}

        rows.append({
            "title": attrs.get("name"),
            "summary": attrs.get("description"),
            "data_kind": "vector",  # default; we can refine later
            "access_kind": "arcgis_hub",
            "homepage_url": links.get("itemPage") or links.get("self"),
            "endpoint_url": links.get("esriRest") or attrs.get("url") or links.get("self"),
            "provider": attrs.get("owner") or attrs.get("orgId"),
            "license": attrs.get("license"),
            "tags": ",".join(attrs.get("tags", [])) if isinstance(attrs.get("tags"), list) else attrs.get("tags"),
            "spatial_extent": pd.NA,
            "temporal_extent": pd.NA,
            "source_name": "arcgis_hub_global",
            "source_url": HOST,
            "hub_dataset_id": rec.get("id"),
            "modified": attrs.get("modified"),
            "orgId": attrs.get("orgId"),
        })

    if len(data) < page_size:
        print(f"STOP: last page {page}")
        break

    page += 1

arcgis_global = pd.DataFrame(rows)

arcgis_global.to_parquet(REG / "arcgis_datasets.parquet", index=False)

arcgis_sources = pd.DataFrame([{
    "source_name": "arcgis_hub_global",
    "source_kind": "arcgis_hub",
    "source_url": HOST,
    "notes": "Global ArcGIS Hub dataset feed (capped at 10k rows to avoid deep-pagination server errors)"
}])
arcgis_sources.to_parquet(REG / "arcgis_hubs.parquet", index=False)

print("WROTE:", REG / "arcgis_datasets.parquet", "rows=", len(arcgis_global))
print("WROTE:", REG / "arcgis_hubs.parquet", "rows=", len(arcgis_sources))

arcgis_global[["title","orgId","homepage_url","endpoint_url","license","tags"]].head(10)

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/arcgis_datasets.parquet rows= 10000
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/arcgis_hubs.parquet rows= 1


,title,orgId,homepage_url,endpoint_url,license,tags
0,NSW Imagery Web Service GDA94,None,https://www.arcgis.com/home/item.html?id=ca937...,https://www.arcgis.com/sharing/content/items/c...,custom,"ADS,ADS Sensor,Aerial Imagery Mosa,Aerial Imag..."
1,Development Plan 2023 2029 RPS Record of Prote...,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=60138...,https://www.arcgis.com/sharing/content/items/6...,CC-BY-4.0,"DP2329,Planning & Cadastral,Planning,Planning ..."
2,Development Plan 2023 2029 Specific Objectives...,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=4dc2e...,https://www.arcgis.com/sharing/content/items/4...,CC-BY-4.0,"DP2329,Planning & Cadastral,Planning,Planning ..."
3,Development Plan 2023 2029 ACA Architectural C...,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=5c7e4...,https://www.arcgis.com/sharing/content/items/5...,CC-BY-4.0,"Planning & Cadastral,Planning,Planning and Lan..."
4,Development Plan 2023 2029 LAP Local Area Plan...,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=d7416...,https://www.arcgis.com/sharing/content/items/d...,CC-BY-4.0,"DP2329,Planning,Planning & Cadastral,Planning ..."
5,DAS Batang toru,8KPMLixcrClQQ9z8,https://www.arcgis.com/home/item.html?id=117de...,https://www.arcgis.com/sharing/content/items/1...,none,"DAS,Sibolga,Sumatra"
6,Dublin Airport Noise Zones Development Plan 20...,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=499ad...,https://www.arcgis.com/sharing/content/items/4...,custom,"Community Information,Development Plan 2023-20..."
7,Cycle Lanes Protected Rivervalley/Hartstown FCC,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=a8c57...,https://www.arcgis.com/sharing/content/items/a...,custom,"Transportation,Transport,Active Travel,Roads a..."
8,Protected Greenway Cycling Lanes FCC,CI1e5PKQXvJgmJK8,https://www.arcgis.com/home/item.html?id=68dc0...,https://www.arcgis.com/sharing/content/items/6...,custom,"Transport,Active Travel,Cycling,Transportation..."
9,Reserveringsgebieden waterberging,dmR647kStmcYa6EN,https://www.arcgis.com/home/item.html?id=65998...,https://www.arcgis.com/sharing/content/items/6...,custom,"Kunstwerken,Watersysteem,Waterbeheerplan 5,Bel..."


In [26]:
import pandas as pd

arc = pd.read_parquet(REG / "arcgis_datasets.parquet")

print("arcgis_datasets:", arc.shape)
print("Unique hub_dataset_id:", arc["hub_dataset_id"].nunique())
print("Duplicate hub_dataset_id rows:", arc.shape[0] - arc["hub_dataset_id"].nunique())

key_fields = ["title","homepage_url","endpoint_url","license","tags","orgId"]
print("\nNull rates:")
display((arc[key_fields].isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("percent_null"))

print("\nEndpoint URL host sample:")
arc["endpoint_host"] = arc["endpoint_url"].fillna("").str.extract(r"^https?://([^/]+)/", expand=False)
display(arc["endpoint_host"].value_counts().head(15))

arcgis_datasets: (10000, 16)
Unique hub_dataset_id: 9950
Duplicate hub_dataset_id rows: 50

Null rates:


,percent_null
orgId,29.72
license,0.01
tags,0.01
title,0.00
homepage_url,0.00
endpoint_url,0.00



Endpoint URL host sample:


endpoint_host
www.arcgis.com    10000
Name: count, dtype: int64

In [28]:
import pandas as pd

arc = pd.read_parquet(REG / "arcgis_datasets.parquet")

def classify_endpoint(u):
    if u is None or pd.isna(u):
        return "none"
    s = str(u)
    if "/sharing/content/items/" in s:
        return "arcgis_item_json"
    if "/rest/services/" in s:
        return "arcgis_rest_service"
    if s.startswith("http://") or s.startswith("https://"):
        return "http_other"
    return "other"

arc["endpoint_kind"] = arc["endpoint_url"].apply(classify_endpoint)

print(arc["endpoint_kind"].value_counts(dropna=False))
display(arc[["title","endpoint_kind","endpoint_url"]].head(25))

endpoint_kind
arcgis_item_json    10000
Name: count, dtype: int64


,title,endpoint_kind,endpoint_url
0,NSW Imagery Web Service GDA94,arcgis_item_json,https://www.arcgis.com/sharing/content/items/c...
1,Development Plan 2023 2029 RPS Record of Prote...,arcgis_item_json,https://www.arcgis.com/sharing/content/items/6...
2,Development Plan 2023 2029 Specific Objectives...,arcgis_item_json,https://www.arcgis.com/sharing/content/items/4...
3,Development Plan 2023 2029 ACA Architectural C...,arcgis_item_json,https://www.arcgis.com/sharing/content/items/5...
4,Development Plan 2023 2029 LAP Local Area Plan...,arcgis_item_json,https://www.arcgis.com/sharing/content/items/d...
5,DAS Batang toru,arcgis_item_json,https://www.arcgis.com/sharing/content/items/1...
6,Dublin Airport Noise Zones Development Plan 20...,arcgis_item_json,https://www.arcgis.com/sharing/content/items/4...
7,Cycle Lanes Protected Rivervalley/Hartstown FCC,arcgis_item_json,https://www.arcgis.com/sharing/content/items/a...
8,Protected Greenway Cycling Lanes FCC,arcgis_item_json,https://www.arcgis.com/sharing/content/items/6...
9,Reserveringsgebieden waterberging,arcgis_item_json,https://www.arcgis.com/sharing/content/items/6...


In [29]:
import pandas as pd

arc = pd.read_parquet(REG / "arcgis_datasets.parquet")

txt = (
    arc["title"].fillna("") + " " +
    arc["tags"].fillna("") + " " +
    arc["homepage_url"].fillna("")
).str.lower()

# broad keywords first
hits = arc[txt.str.contains(r"\bsangis\b|san gis|san diego|county of san diego|sandag")].copy()

print("Potential SanGIS/San Diego hits:", len(hits))

display(hits[["title","homepage_url","endpoint_url","orgId","tags"]].head(50))

Potential SanGIS/San Diego hits: 0


,title,homepage_url,endpoint_url,orgId,tags


In [30]:
import pandas as pd
import requests

HOST = "https://hub.arcgis.com"
DATASETS_URL = f"{HOST}/api/v3/datasets"
FIELDS = "name,description,url,tags,license,modified,owner,orgId"

def search_keyword(q, pages=3, page_size=50):
    rows = []
    for page in range(1, pages+1):
        params = {
            "fields[datasets]": FIELDS,
            "q": q,                 # keyword search
            "page[number]": page,
            "page[size]": page_size,
            "sort": "-modified",
        }
        r = requests.get(DATASETS_URL, params=params, timeout=60)
        r.raise_for_status()
        js = r.json()
        data = js.get("data", []) or []
        for rec in data:
            a = rec.get("attributes", {}) or {}
            links = rec.get("links", {}) or {}
            rows.append({
                "q": q,
                "title": a.get("name"),
                "homepage_url": links.get("itemPage") or links.get("self"),
                "endpoint_url": links.get("esriRest") or a.get("url") or links.get("self"),
                "orgId": a.get("orgId"),
                "owner": a.get("owner"),
                "tags": ",".join(a.get("tags", [])) if isinstance(a.get("tags"), list) else a.get("tags"),
                "modified": a.get("modified"),
                "hub_dataset_id": rec.get("id"),
            })
        if len(data) < page_size:
            break
    return pd.DataFrame(rows)

sangis_search = search_keyword("sangis", pages=5, page_size=50)
print("Keyword search rows:", sangis_search.shape)
display(sangis_search.head(50))

Keyword search rows: (250, 9)


,q,title,homepage_url,endpoint_url,orgId,owner,tags,modified,hub_dataset_id
0,sangis,Documents,https://www.arcgis.com/home/item.html?id=15f0e...,https://www.arcgis.com/sharing/content/items/1...,None,louisewedleySanGIS,"SanGIS Website,SanGIS Documents",1772227543000,15f0e653cb294e9eaac20a527b0316a1
1,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=ccf52...,https://www.arcgis.com/sharing/content/items/c...,eGSDp8lpKe5izqVc,nsadatmand_UCSDOnline,,1772227123000,ccf522887d61493e93b7cb6b78ffeed4
2,sangis,SanGIS Website,https://www.arcgis.com/home/item.html?id=54993...,https://www.arcgis.com/sharing/content/items/5...,None,louisewedleySanGIS,Hub Site,1772226610000,549936191bce4467babb77963cb7f6f2
3,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=2b858...,https://www.arcgis.com/sharing/content/items/2...,eGSDp8lpKe5izqVc,dialvarez_UCSDOnline,,1772153112000,2b858ffad5234ebf801b07205b983ab4
4,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=3d70b...,https://www.arcgis.com/sharing/content/items/3...,eGSDp8lpKe5izqVc,eriveraochoa_UCSDOnline,,1772153105000,3d70b5de8984463f8d2c943c7792f95e
5,sangis,SD Address Points,https://www.arcgis.com/home/item.html?id=812ea...,https://www.arcgis.com/sharing/content/items/8...,None,akeiper_lmh,"Addresses,Parcel Address,SITUS Address,Assesso...",1772144316807,812eaefda212482197bf71b48d03988e_0
6,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=b127c...,https://www.arcgis.com/sharing/content/items/b...,eGSDp8lpKe5izqVc,jtb003_UCSDOnline,,1772075799000,b127cb6f10044aea96d1029c5cc2e1e8
7,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=64c0c...,https://www.arcgis.com/sharing/content/items/6...,eGSDp8lpKe5izqVc,cml022_UCSDOnline,,1772075668000,64c0c11f860444ed87689a2533454abc
8,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=f063e...,https://www.arcgis.com/sharing/content/items/f...,eGSDp8lpKe5izqVc,c9reyes_UCSDOnline,,1772073067000,f063eb159f624fd193c17fca23835b0a
9,sangis,Nursing Homes and Park Access in San Diego,https://www.arcgis.com/home/item.html?id=dbb9d...,https://www.arcgis.com/sharing/content/items/d...,eGSDp8lpKe5izqVc,hbernstein_UCSDOnline,,1772072955000,dbb9d5c6cf854b9c814f611f3308838f


In [31]:
import pandas as pd

p = REG / "arcgis_datasets.parquet"
arc = pd.read_parquet(p)

before = len(arc)
arc = arc.drop_duplicates(subset=["hub_dataset_id"]).reset_index(drop=True)
after = len(arc)

arc.to_parquet(p, index=False)
print("Deduped arcgis_datasets.parquet:", before, "->", after)

Deduped arcgis_datasets.parquet: 10000 -> 9950


In [32]:
import pandas as pd

# assumes sangis_search exists from your D4 cell
sangis_search["source_name"] = "arcgis_keyword_sangis"
sangis_search["source_url"]  = "https://hub.arcgis.com"
sangis_search.to_parquet(REG / "arcgis_sangis_seed.parquet", index=False)

print("WROTE:", REG / "arcgis_sangis_seed.parquet", "rows=", len(sangis_search))
sangis_search[["title","owner","homepage_url","endpoint_url"]].head(25)

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/arcgis_sangis_seed.parquet rows= 250


,title,owner,homepage_url,endpoint_url
0,Documents,louisewedleySanGIS,https://www.arcgis.com/home/item.html?id=15f0e...,https://www.arcgis.com/sharing/content/items/1...
1,Nursing Homes and Park Access in San Diego,nsadatmand_UCSDOnline,https://www.arcgis.com/home/item.html?id=ccf52...,https://www.arcgis.com/sharing/content/items/c...
2,SanGIS Website,louisewedleySanGIS,https://www.arcgis.com/home/item.html?id=54993...,https://www.arcgis.com/sharing/content/items/5...
3,Nursing Homes and Park Access in San Diego,dialvarez_UCSDOnline,https://www.arcgis.com/home/item.html?id=2b858...,https://www.arcgis.com/sharing/content/items/2...
4,Nursing Homes and Park Access in San Diego,eriveraochoa_UCSDOnline,https://www.arcgis.com/home/item.html?id=3d70b...,https://www.arcgis.com/sharing/content/items/3...
5,SD Address Points,akeiper_lmh,https://www.arcgis.com/home/item.html?id=812ea...,https://www.arcgis.com/sharing/content/items/8...
6,Nursing Homes and Park Access in San Diego,jtb003_UCSDOnline,https://www.arcgis.com/home/item.html?id=b127c...,https://www.arcgis.com/sharing/content/items/b...
7,Nursing Homes and Park Access in San Diego,cml022_UCSDOnline,https://www.arcgis.com/home/item.html?id=64c0c...,https://www.arcgis.com/sharing/content/items/6...
8,Nursing Homes and Park Access in San Diego,c9reyes_UCSDOnline,https://www.arcgis.com/home/item.html?id=f063e...,https://www.arcgis.com/sharing/content/items/f...
9,Nursing Homes and Park Access in San Diego,hbernstein_UCSDOnline,https://www.arcgis.com/home/item.html?id=dbb9d...,https://www.arcgis.com/sharing/content/items/d...


In [33]:
# ============================================================
# CELL E — Merge ArcGIS global registry into master datasets/sources
# ============================================================

import pandas as pd
import hashlib

def stable_id(*parts) -> str:
    s = "|".join([("" if p is None or pd.isna(p) else str(p)) for p in parts])
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

master = pd.read_parquet(REG / "datasets.parquet")
arcgis = pd.read_parquet(REG / "arcgis_datasets.parquet").copy()

# Dedup ArcGIS on hub id (just in case)
arcgis = arcgis.drop_duplicates(subset=["hub_dataset_id"]).reset_index(drop=True)

# Build canonical arcgis rows matching master schema
canon = list(master.columns)
for c in canon:
    if c not in arcgis.columns:
        arcgis[c] = pd.NA

arcgis = arcgis[canon + ["hub_dataset_id"]].copy()

# For ArcGIS, hub_dataset_id is the strongest identifier
arcgis["dataset_id"] = [
    stable_id(r["source_name"], r["hub_dataset_id"])
    for _, r in arcgis.iterrows()
]

arcgis = arcgis[canon].copy()

combined = pd.concat([master, arcgis], ignore_index=True)
combined = combined.drop_duplicates(subset=["dataset_id"]).reset_index(drop=True)
combined.to_parquet(REG / "datasets.parquet", index=False)

# Merge sources
sources = pd.read_parquet(REG / "sources.parquet")

arc_sources = pd.DataFrame([{
    "source_name": "arcgis_hub_global",
    "source_kind": "arcgis_hub",
    "source_url": "https://hub.arcgis.com",
    "notes": "Global ArcGIS Hub dataset feed (top 10k recent, deduped)"
}])

# Ensure columns exist
if "source_kind" not in sources.columns:
    sources["source_kind"] = "opengeos_tsv"
if "notes" not in sources.columns:
    sources["notes"] = ""

sources2 = pd.concat([
    sources[["source_name","source_kind","source_url","notes"]],
    arc_sources[["source_name","source_kind","source_url","notes"]],
], ignore_index=True).drop_duplicates(subset=["source_name","source_url"])

sources2.to_parquet(REG / "sources.parquet", index=False)

print("MASTER datasets rows:", len(combined))
print("MASTER sources rows:", len(sources2))
print("\nCounts by access_kind:")
print(combined["access_kind"].value_counts())

MASTER datasets rows: 34139
MASTER sources rows: 8

Counts by access_kind:
access_kind
cmr              20592
arcgis_hub        9950
aws_open_data     2523
gee                835
stac_catalog       239
Name: count, dtype: int64


/var/folders/fd/bjbtdt71459d1zh1mb1700mw0000gn/T/ipykernel_84100/3354602385.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([master, arcgis], ignore_index=True)


In [34]:
# ============================================================
# CELL O1 — Overture registry (Phase 1)
# ============================================================

import pandas as pd
import hashlib

def stable_id(*parts) -> str:
    s = "|".join([("" if p is None or pd.isna(p) else str(p)) for p in parts])
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

# Official endpoints (from Overture docs)
OVERTURE_STAC_ROOT = "https://stac.overturemaps.org/catalog.json"
OVERTURE_S3_ROOT   = "s3://overturemaps-us-west-2/release/<RELEASE>"
OVERTURE_AZ_ROOT   = "https://overturemapswestus2.blob.core.windows.net/release/<RELEASE>"
OVERTURE_LATEST    = "2026-02-18.0/"  # from docs (informational)

rows = [
    {
        "title": "Overture Maps — STAC Releases Catalog",
        "summary": "STAC catalog of Overture releases; use it to discover the latest release and browse themes/types.",
        "data_kind": "vector",
        "access_kind": "overture_stac",
        "homepage_url": "https://docs.overturemaps.org/getting-data/",
        "endpoint_url": OVERTURE_STAC_ROOT,
        "provider": "Overture Maps Foundation",
        "license": "see Overture docs",
        "tags": "overture,stac,geoparquet,vector,open-data",
        "spatial_extent": pd.NA,
        "temporal_extent": pd.NA,
        "source_name": "overture",
        "source_url": "https://docs.overturemaps.org/getting-data/",
    },
    {
        "title": "Overture Maps — AWS S3 Release Root",
        "summary": f"Release GeoParquet on S3. Latest release per docs: {OVERTURE_LATEST}",
        "data_kind": "vector",
        "access_kind": "overture_s3",
        "homepage_url": "https://docs.overturemaps.org/getting-data/",
        "endpoint_url": OVERTURE_S3_ROOT,
        "provider": "Overture Maps Foundation",
        "license": "see Overture docs",
        "tags": "overture,s3,geoparquet,vector,open-data",
        "spatial_extent": pd.NA,
        "temporal_extent": pd.NA,
        "source_name": "overture",
        "source_url": "https://docs.overturemaps.org/getting-data/",
    },
    {
        "title": "Overture Maps — Azure Release Root",
        "summary": f"Release GeoParquet on Azure Blob. Latest release per docs: {OVERTURE_LATEST}",
        "data_kind": "vector",
        "access_kind": "overture_azure",
        "homepage_url": "https://docs.overturemaps.org/getting-data/",
        "endpoint_url": OVERTURE_AZ_ROOT,
        "provider": "Overture Maps Foundation",
        "license": "see Overture docs",
        "tags": "overture,azure,geoparquet,vector,open-data",
        "spatial_extent": pd.NA,
        "temporal_extent": pd.NA,
        "source_name": "overture",
        "source_url": "https://docs.overturemaps.org/getting-data/",
    },
]

overture = pd.DataFrame(rows)

# dataset_id
overture["dataset_id"] = [
    stable_id(r["source_name"], r["access_kind"], r["endpoint_url"])
    for _, r in overture.iterrows()
]

# Column order should match your master registry
master = pd.read_parquet(REG / "datasets.parquet")
canon = list(master.columns)
for c in canon:
    if c not in overture.columns:
        overture[c] = pd.NA
overture = overture[canon]

overture.to_parquet(REG / "overture_datasets.parquet", index=False)

sources = pd.DataFrame([{
    "source_name": "overture",
    "source_kind": "official_docs",
    "source_url": "https://docs.overturemaps.org/getting-data/",
    "notes": "Overture official docs + STAC release catalog"
}])
sources.to_parquet(REG / "overture_sources.parquet", index=False)

print("WROTE:", REG / "overture_datasets.parquet", "rows=", len(overture))
print("WROTE:", REG / "overture_sources.parquet", "rows=", len(sources))
overture

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_datasets.parquet rows= 3
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_sources.parquet rows= 1


,dataset_id,title,summary,data_kind,access_kind,homepage_url,endpoint_url,provider,license,tags,spatial_extent,temporal_extent,source_name,source_url
0,6cfecc50ae043e1d,Overture Maps — STAC Releases Catalog,STAC catalog of Overture releases; use it to d...,vector,overture_stac,https://docs.overturemaps.org/getting-data/,https://stac.overturemaps.org/catalog.json,Overture Maps Foundation,see Overture docs,"overture,stac,geoparquet,vector,open-data",<NA>,<NA>,overture,https://docs.overturemaps.org/getting-data/
1,14d3ff00fc140dc3,Overture Maps — AWS S3 Release Root,Release GeoParquet on S3. Latest release per d...,vector,overture_s3,https://docs.overturemaps.org/getting-data/,s3://overturemaps-us-west-2/release/<RELEASE>,Overture Maps Foundation,see Overture docs,"overture,s3,geoparquet,vector,open-data",<NA>,<NA>,overture,https://docs.overturemaps.org/getting-data/
2,24e65eb583dcb4c2,Overture Maps — Azure Release Root,Release GeoParquet on Azure Blob. Latest relea...,vector,overture_azure,https://docs.overturemaps.org/getting-data/,https://overturemapswestus2.blob.core.windows....,Overture Maps Foundation,see Overture docs,"overture,azure,geoparquet,vector,open-data",<NA>,<NA>,overture,https://docs.overturemaps.org/getting-data/


In [36]:
import pandas as pd

o = pd.read_parquet(REG / "overture_datasets.parquet")

print(o[["access_kind","endpoint_url"]])

print("\nChecks:")
print("Has STAC catalog.json:",
      (o["endpoint_url"].str.contains("stac.overturemaps.org/catalog.json", na=False)).any())

print("Has S3 release root placeholder:",
      (o["endpoint_url"].str.contains("s3://overturemaps-us-west-2/release/<RELEASE>", na=False)).any())

print("Has Azure release root placeholder:",
      (o["endpoint_url"].str.contains("blob.core.windows.net/release/<RELEASE>", na=False)).any())

      access_kind                                       endpoint_url
0   overture_stac         https://stac.overturemaps.org/catalog.json
1     overture_s3      s3://overturemaps-us-west-2/release/<RELEASE>
2  overture_azure  https://overturemapswestus2.blob.core.windows....

Checks:
Has STAC catalog.json: True
Has S3 release root placeholder: True
Has Azure release root placeholder: True


In [37]:
# ============================================================
# CELL O2 — Merge Overture rows into master datasets/sources
# ============================================================

import pandas as pd

master = pd.read_parquet(REG / "datasets.parquet")
src_ds = pd.read_parquet(REG / "overture_datasets.parquet")

combined = pd.concat([master, src_ds], ignore_index=True)
combined = combined.drop_duplicates(subset=["dataset_id"]).reset_index(drop=True)
combined.to_parquet(REG / "datasets.parquet", index=False)

sources = pd.read_parquet(REG / "sources.parquet")
src_sources = pd.read_parquet(REG / "overture_sources.parquet")

# ensure schema
if "source_kind" not in sources.columns:
    sources["source_kind"] = "opengeos_tsv"
if "notes" not in sources.columns:
    sources["notes"] = ""

sources2 = pd.concat(
    [sources[["source_name","source_kind","source_url","notes"]], src_sources],
    ignore_index=True
).drop_duplicates(subset=["source_name","source_url"]).reset_index(drop=True)

sources2.to_parquet(REG / "sources.parquet", index=False)

print("MASTER datasets rows:", len(combined))
print("MASTER sources rows:", len(sources2))
print("\nCounts by access_kind (top):")
print(combined["access_kind"].value_counts().head(20))

MASTER datasets rows: 34142
MASTER sources rows: 9

Counts by access_kind (top):
access_kind
cmr               20592
arcgis_hub         9950
aws_open_data      2523
gee                 835
stac_catalog        239
overture_stac         1
overture_s3           1
overture_azure        1
Name: count, dtype: int64


/var/folders/fd/bjbtdt71459d1zh1mb1700mw0000gn/T/ipykernel_84100/4229640308.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([master, src_ds], ignore_index=True)


In [38]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

latest = con.execute("""
  SELECT latest
  FROM 'https://stac.overturemaps.org/catalog.json'
""").fetchone()[0]

latest

'2026-02-18.0'

In [39]:
import json, urllib.request
from urllib.parse import urljoin

ROOT = "https://stac.overturemaps.org/catalog.json"

with urllib.request.urlopen(ROOT) as f:
    cat = json.load(f)

children = [l for l in cat.get("links", []) if l.get("rel") == "child"]

print("Root id:", cat.get("id"))
print("Child catalogs (likely releases):", len(children))
for l in children[:50]:
    print("-", l.get("title") or "", "=>", l.get("href"))

Root id: Overture Releases
Child catalogs (likely releases): 2
- Latest Overture Release => ./2026-02-18.0/catalog.json
- 2026-01-21.0 Overture Release => ./2026-01-21.0/catalog.json


In [41]:
# ============================================================
# CELL O4 — Build Overture inventory for one release (latest)
# Fix: resolves relative hrefs like ./2026-02-18.0/catalog.json
# Writes: open-data/registry/overture_inventory.parquet
# ============================================================

import json
import urllib.request
from urllib.parse import urljoin
import pandas as pd
import hashlib

ROOT = "https://stac.overturemaps.org/catalog.json"
LATEST_RELEASE = "2026-02-18.0"  # confirmed by your duckdb smoke test

def stable_id(*parts) -> str:
    s = "|".join([("" if p is None else str(p)) for p in parts])
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def fetch_json(url: str) -> dict:
    with urllib.request.urlopen(url) as f:
        return json.load(f)

def abs_href(base: str, href: str) -> str:
    # urljoin correctly handles ./ relative links
    return urljoin(base, href)

root = fetch_json(ROOT)

# --- find the release link (child) and resolve it to absolute URL ---
release_href_rel = None
for l in root.get("links", []):
    if l.get("rel") == "child" and (LATEST_RELEASE in (l.get("href") or "") or LATEST_RELEASE in (l.get("title") or "")):
        release_href_rel = l.get("href")
        break

if release_href_rel is None:
    for l in root.get("links", []):
        if l.get("rel") == "child" and LATEST_RELEASE in (l.get("href") or ""):
            release_href_rel = l.get("href")
            break

if release_href_rel is None:
    raise RuntimeError(f"Could not find release link for {LATEST_RELEASE} in root catalog links.")

release_href = abs_href(ROOT, release_href_rel)

rel = fetch_json(release_href)

# --- inventory: list child/collection links under the release catalog ---
rows = []
for l in rel.get("links", []):
    if l.get("rel") in ("child", "collection"):
        href_abs = abs_href(release_href, l.get("href"))
        rows.append({
            "inventory_id": stable_id("overture", LATEST_RELEASE, href_abs),
            "source_name": "overture",
            "release": LATEST_RELEASE,
            "kind": l.get("rel"),
            "title": l.get("title"),
            "href": href_abs,
        })

inv = pd.DataFrame(rows).drop_duplicates(subset=["inventory_id"]).reset_index(drop=True)

out_path = REG / "overture_inventory.parquet"
inv.to_parquet(out_path, index=False)

print("Root:", ROOT)
print("Release catalog:", release_href)
print("WROTE:", out_path, "rows=", len(inv))

inv.head(30)

Root: https://stac.overturemaps.org/catalog.json
Release catalog: https://stac.overturemaps.org/2026-02-18.0/catalog.json
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_inventory.parquet rows= 6


,inventory_id,source_name,release,kind,title,href
0,e7e2399e3723cb32,overture,2026-02-18.0,child,divisions,https://stac.overturemaps.org/2026-02-18.0/div...
1,732f56d89900e9bd,overture,2026-02-18.0,child,addresses,https://stac.overturemaps.org/2026-02-18.0/add...
2,7a177f4f39090642,overture,2026-02-18.0,child,places,https://stac.overturemaps.org/2026-02-18.0/pla...
3,f8653663ceab610f,overture,2026-02-18.0,child,transportation,https://stac.overturemaps.org/2026-02-18.0/tra...
4,c743d072f8312b0e,overture,2026-02-18.0,child,buildings,https://stac.overturemaps.org/2026-02-18.0/bui...
5,8e7eaed9aaad0bdf,overture,2026-02-18.0,child,base,https://stac.overturemaps.org/2026-02-18.0/bas...


In [44]:
import json, urllib.request
from urllib.parse import urljoin
import pandas as pd

BUILDINGS_CAT = "https://stac.overturemaps.org/2026-02-18.0/buildings/catalog.json"

with urllib.request.urlopen(BUILDINGS_CAT) as f:
    bcat = json.load(f)

children = [l for l in bcat.get("links", []) if l.get("rel") == "child"]
rows = []
for i, l in enumerate(children):
    rows.append({
        "i": i,
        "title": l.get("title"),
        "href": urljoin(BUILDINGS_CAT, l.get("href")),
        "type": l.get("type"),
    })

df_children = pd.DataFrame(rows)
df_children

,i,title,href,type
0,0,None,https://stac.overturemaps.org/2026-02-18.0/bui...,application/json
1,1,None,https://stac.overturemaps.org/2026-02-18.0/bui...,application/json


In [45]:
import json, urllib.request
from urllib.parse import urljoin
import pandas as pd

BUILDINGS_CAT = "https://stac.overturemaps.org/2026-02-18.0/buildings/catalog.json"

def fetch(url):
    with urllib.request.urlopen(url) as f:
        return json.load(f)

bcat = fetch(BUILDINGS_CAT)
children = [urljoin(BUILDINGS_CAT, l.get("href")) for l in bcat.get("links", []) if l.get("rel") == "child"]

out = []
for href in children:
    obj = fetch(href)
    links = obj.get("links", []) or []
    assets = obj.get("assets", {}) or {}

    out.append({
        "href": href,
        "type": obj.get("type"),
        "id": obj.get("id"),
        "title": obj.get("title"),
        "n_links": len(links),
        "n_item_links": sum(1 for l in links if l.get("rel") == "item"),
        "n_child_links": sum(1 for l in links if l.get("rel") == "child"),
        "n_assets": len(assets),
        "has_parquet_asset": any(
            (a.get("href","") or "").endswith(".parquet") or ("parquet" in (a.get("type","") or "").lower())
            for a in assets.values()
        )
    })

pd.DataFrame(out)

,href,type,id,title,n_links,n_item_links,n_child_links,n_assets,has_parquet_asset
0,https://stac.overturemaps.org/2026-02-18.0/bui...,Collection,building,None,239,237,0,0,False
1,https://stac.overturemaps.org/2026-02-18.0/bui...,Collection,building_part,None,3,1,0,0,False


In [46]:
import json, urllib.request
from urllib.parse import urljoin
import pandas as pd

def fetch(url):
    with urllib.request.urlopen(url) as f:
        return json.load(f)

def first_item_href(collection_url: str):
    obj = fetch(collection_url)
    for l in obj.get("links", []) or []:
        if l.get("rel") == "item":
            return urljoin(collection_url, l.get("href"))
    return None

# Use the building collection you already opened:
COLLECTION_URL = "https://stac.overturemaps.org/2026-02-18.0/buildings/building/collection.json"

item_href = first_item_href(COLLECTION_URL)
if item_href is None:
    raise RuntimeError("No explicit item links found on the collection. We need to follow child links instead.")

item = fetch(item_href)

assets = item.get("assets", {}) or {}
rows = []
for k, a in assets.items():
    rows.append({
        "asset_key": k,
        "href": a.get("href"),
        "type": a.get("type"),
        "roles": ",".join(a.get("roles", [])) if isinstance(a.get("roles"), list) else a.get("roles"),
        "title": a.get("title"),
    })

adf = pd.DataFrame(rows)

# parquet detection safe even if empty
if len(adf) == 0:
    print("Item has no assets.")
else:
    adf["is_parquet"] = (
        adf["href"].fillna("").str.contains(r"\.parquet(\?|$)", regex=True) |
        adf["type"].fillna("").str.contains("parquet", case=False)
    )
    display(adf.sort_values(["is_parquet","asset_key"], ascending=[False, True]).head(50))

print("Item href:", item_href)
print("Item id:", item.get("id"))

/var/folders/fd/bjbtdt71459d1zh1mb1700mw0000gn/T/ipykernel_84100/2686747532.py:43: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  adf["href"].fillna("").str.contains(r"\.parquet(\?|$)", regex=True) |


,asset_key,href,type,roles,title,is_parquet
0,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,None,None,True
1,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,None,None,True


Item href: https://stac.overturemaps.org/2026-02-18.0/buildings/building/00000/00000.json
Item id: 00000


In [47]:
# ============================================================
# CELL O6 — Build Overture file inventory (GeoParquet hrefs)
# Writes: open-data/registry/overture_files.parquet
# ============================================================

import json, urllib.request
from urllib.parse import urljoin
import pandas as pd
import hashlib

RELEASE = "2026-02-18.0"

def fetch(url):
    with urllib.request.urlopen(url) as f:
        return json.load(f)

def stable_id(*parts) -> str:
    s = "|".join([("" if p is None or pd.isna(p) else str(p)) for p in parts])
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def collect_files_from_collection(collection_url: str, theme: str, layer: str):
    col = fetch(collection_url)
    out = []

    for l in col.get("links", []) or []:
        if l.get("rel") != "item":
            continue

        item_url = urljoin(collection_url, l.get("href"))
        item = fetch(item_url)
        item_id = item.get("id")

        assets = item.get("assets", {}) or {}
        for asset_key, a in assets.items():
            href = a.get("href")
            typ  = a.get("type")

            # Only keep parquet assets
            is_parq = (
                (href or "").endswith(".parquet") or
                ("parquet" in (typ or "").lower())
            )
            if not is_parq:
                continue

            storage = "aws" if asset_key.lower() == "aws" else ("azure" if asset_key.lower() == "azure" else "other")

            out.append({
                "file_id": stable_id("overture", RELEASE, theme, layer, item_id, asset_key, href),
                "source_name": "overture",
                "release": RELEASE,
                "theme": theme,
                "layer": layer,
                "item_id": item_id,
                "asset_key": asset_key,
                "storage": storage,
                "href": href,
                "type": typ,
            })

    return pd.DataFrame(out)

# Collections we discovered
building_collection      = "https://stac.overturemaps.org/2026-02-18.0/buildings/building/collection.json"
building_part_collection = "https://stac.overturemaps.org/2026-02-18.0/buildings/building_part/collection.json"

df1 = collect_files_from_collection(building_collection, theme="buildings", layer="building")
df2 = collect_files_from_collection(building_part_collection, theme="buildings", layer="building_part")

files = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset=["file_id"]).reset_index(drop=True)

out_path = REG / "overture_files.parquet"
files.to_parquet(out_path, index=False)

print("WROTE:", out_path, "rows=", len(files))
print("\nCounts by layer:")
print(files["layer"].value_counts(dropna=False))
print("\nCounts by storage:")
print(files["storage"].value_counts(dropna=False))

files.head(20)

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_files.parquet rows= 476

Counts by layer:
layer
building         474
building_part      2
Name: count, dtype: int64

Counts by storage:
storage
aws      238
azure    238
Name: count, dtype: int64


,file_id,source_name,release,theme,layer,item_id,asset_key,storage,href,type
0,6ef7823d9ddefd40,overture,2026-02-18.0,buildings,building,00000,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet
1,1606bec51c62f9cc,overture,2026-02-18.0,buildings,building,00000,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet
2,416ca661c64883a1,overture,2026-02-18.0,buildings,building,00001,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet
3,b98f9288e589e1a5,overture,2026-02-18.0,buildings,building,00001,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet
4,973845febe2c75fc,overture,2026-02-18.0,buildings,building,00002,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet
5,cd5d234a3d4dbee0,overture,2026-02-18.0,buildings,building,00002,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet
6,8b1b6d9cf7985e26,overture,2026-02-18.0,buildings,building,00003,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet
7,bf9a6d52028e503b,overture,2026-02-18.0,buildings,building,00003,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet
8,e63a2d8e71dca084,overture,2026-02-18.0,buildings,building,00004,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet
9,c54787ae36e86935,overture,2026-02-18.0,buildings,building,00004,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet


In [48]:
import duckdb
import pandas as pd

files = pd.read_parquet(REG / "overture_files.parquet")

# pick first AWS parquet from buildings/building
href = files[(files["layer"]=="building") & (files["storage"]=="aws")]["href"].iloc[0]
print("Using parquet:", href)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL spatial; LOAD spatial;")

# Show columns (quick)
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{href}')").df()
display(cols)

# Sample rows
df = con.execute(f"SELECT * FROM read_parquet('{href}') LIMIT 5").df()
df

Using parquet: https://overturemaps-us-west-2.s3.us-west-2.amazonaws.com/release/2026-02-18.0/theme=buildings/type=building/part-00000-5fef4c0a-b212-4641-abd0-d300e9f25c51-c000.zstd.parquet


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,geometry,GEOMETRY,YES,None,None,None
2,bbox,"STRUCT(xmin FLOAT, xmax FLOAT, ymin FLOAT, yma...",YES,None,None,None
3,version,INTEGER,YES,None,None,None
4,sources,"STRUCT(property VARCHAR, dataset VARCHAR, lice...",YES,None,None,None
5,level,INTEGER,YES,None,None,None
6,subtype,VARCHAR,YES,None,None,None
7,class,VARCHAR,YES,None,None,None
8,height,DOUBLE,YES,None,None,None
9,names,"STRUCT(""primary"" VARCHAR, common MAP(VARCHAR, ...",YES,None,None,None


,id,geometry,bbox,version,sources,level,subtype,class,height,names,...,facade_color,facade_material,roof_material,roof_shape,roof_direction,roof_orientation,roof_color,roof_height,theme,type
0,68c64c74-0704-4030-8d79-bb30b20fd032,"[2, 4, 0, 0, 0, 0, 0, 0, 111, 102, 39, 195, 20...","{'xmin': -167.40013122558594, 'xmax': -167.399...",1,"[{'property': '', 'dataset': 'OpenStreetMap', ...",<NA>,None,None,NaN,"{'primary': 'Marlene Weather Station', 'common...",...,None,None,None,None,NaN,None,None,NaN,buildings,building
1,34adb9da-7e11-4cc9-b5a6-53ae5518188b,"[2, 4, 0, 0, 0, 0, 0, 0, 16, 118, 10, 195, 214...","{'xmin': -138.461181640625, 'xmax': -138.45275...",1,"[{'property': '', 'dataset': 'OpenStreetMap', ...",<NA>,None,None,NaN,<NA>,...,None,None,None,None,NaN,None,None,NaN,buildings,building
2,1b57ffcd-89ac-40b7-a02a-2182a490bde1,"[2, 4, 0, 0, 0, 0, 0, 0, 250, 204, 8, 195, 93,...","{'xmin': -136.80068969726562, 'xmax': -136.800...",1,"[{'property': '', 'dataset': 'OpenStreetMap', ...",<NA>,None,None,NaN,<NA>,...,None,None,None,None,NaN,None,None,NaN,buildings,building
3,28a13967-5dc3-4fdb-b0ae-c104df807efc,"[2, 4, 0, 0, 0, 0, 0, 0, 81, 205, 8, 195, 249,...","{'xmin': -136.80201721191406, 'xmax': -136.801...",1,"[{'property': '', 'dataset': 'OpenStreetMap', ...",<NA>,None,None,NaN,<NA>,...,None,None,None,None,NaN,None,None,NaN,buildings,building
4,4a9dd4fc-a1f2-4b95-aef9-5baaf93583f9,"[2, 4, 0, 0, 0, 0, 0, 0, 24, 205, 8, 195, 226,...","{'xmin': -136.8011474609375, 'xmax': -136.8007...",1,"[{'property': '', 'dataset': 'OpenStreetMap', ...",<NA>,None,None,NaN,<NA>,...,None,None,None,None,NaN,None,None,NaN,buildings,building


In [4]:
minx, miny, maxx, maxy = -117.30, 32.60, -117.00, 32.95

In [3]:
import duckdb
import pandas as pd

files = pd.read_parquet(REG / "overture_files.parquet")
href = files[(files["layer"]=="building") & (files["storage"]=="aws")]["href"].iloc[0]

minx, miny, maxx, maxy = -117.30, 32.60, -117.00, 32.95

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL spatial; LOAD spatial;")

sql = f"""
SELECT
  id,
  subtype,
  class,
  height,
  num_floors,
  bbox
FROM read_parquet('{href}')
WHERE ST_Intersects(
  geometry,
  ST_MakeEnvelope({minx}, {miny}, {maxx}, {maxy})
)
LIMIT 50
"""

df = con.execute(sql).df()
print("matches:", len(df))
df.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

matches: 0


,id,subtype,class,height,num_floors,bbox


In [5]:
import duckdb
import pandas as pd

files = pd.read_parquet(REG / "overture_files.parquet")
aws_parts = files[(files["layer"]=="building") & (files["storage"]=="aws")]["href"].tolist()
print("AWS parts:", len(aws_parts))

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("PRAGMA enable_progress_bar=false;")
con.execute("PRAGMA threads=4;")  # keep sane; adjust if you want
con.execute("PRAGMA temp_directory='/tmp';")  # IMPORTANT: don't nuke your home drive

# Build a per-file bbox summary (min/max across bbox struct)
# DuckDB exposes filename via filename=true in read_parquet.
url_list_sql = ",\n".join([f"'{u}'" for u in aws_parts])

sql = f"""
WITH src AS (
  SELECT
    filename,
    bbox.xmin AS xmin,
    bbox.ymin AS ymin,
    bbox.xmax AS xmax,
    bbox.ymax AS ymax
  FROM read_parquet([{url_list_sql}], filename=true)
)
SELECT
  filename,
  min(xmin) AS xmin,
  min(ymin) AS ymin,
  max(xmax) AS xmax,
  max(ymax) AS ymax
FROM src
GROUP BY filename
"""

part_bbox = con.execute(sql).df()
print("parts summarized:", len(part_bbox))

# save index for reuse (this is your "smart indexing" artifact)
out = REG / "overture_buildings_part_bbox.parquet"
part_bbox.to_parquet(out, index=False)
print("WROTE:", out)

part_bbox.head(10)

AWS parts: 237
parts summarized: 237
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_buildings_part_bbox.parquet


,filename,xmin,ymin,xmax,ymax
0,https://overturemaps-us-west-2.s3.us-west-2.am...,-71.666977,-33.750317,-56.249943,-28.124910
1,https://overturemaps-us-west-2.s3.us-west-2.am...,-49.219048,-26.015821,-44.999912,-22.498224
2,https://overturemaps-us-west-2.s3.us-west-2.am...,-67.500046,-16.171896,-44.999844,0.000190
3,https://overturemaps-us-west-2.s3.us-west-2.am...,-101.250107,15.468680,-92.812408,19.687960
4,https://overturemaps-us-west-2.s3.us-west-2.am...,-99.843842,16.874899,-95.624916,19.687685
5,https://overturemaps-us-west-2.s3.us-west-2.am...,-108.280945,22.499887,-96.800995,30.937717
6,https://overturemaps-us-west-2.s3.us-west-2.am...,-112.500168,28.124651,-96.329475,39.375523
7,https://overturemaps-us-west-2.s3.us-west-2.am...,-123.750175,33.749538,-112.499802,43.593830
8,https://overturemaps-us-west-2.s3.us-west-2.am...,-9.544483,33.750080,0.000333,43.786839
9,https://overturemaps-us-west-2.s3.us-west-2.am...,-75.938377,39.374260,-70.526138,42.188408


In [6]:
import pandas as pd

part_bbox = pd.read_parquet(REG / "overture_buildings_part_bbox.parquet")

minx, miny, maxx, maxy = -117.30, 32.60, -117.00, 32.95

# bbox intersection test (no geometry)
cand = part_bbox[
    (part_bbox["xmax"] >= minx) &
    (part_bbox["xmin"] <= maxx) &
    (part_bbox["ymax"] >= miny) &
    (part_bbox["ymin"] <= maxy)
].copy()

print("Candidate parts for bbox:", len(cand))
cand.head(20)

Candidate parts for bbox: 2


,filename,xmin,ymin,xmax,ymax
124,https://overturemaps-us-west-2.s3.us-west-2.am...,-119.572212,17.456800,-101.249466,33.750771
187,https://overturemaps-us-west-2.s3.us-west-2.am...,-179.798996,23.868008,-90.102760,78.788376


In [7]:
import duckdb
import pandas as pd

cand = pd.read_parquet(REG / "overture_buildings_part_bbox.parquet")
minx, miny, maxx, maxy = -117.30, 32.60, -117.00, 32.95
cand = cand[
    (cand["xmax"] >= minx) &
    (cand["xmin"] <= maxx) &
    (cand["ymax"] >= miny) &
    (cand["ymin"] <= maxy)
].copy()

urls = cand["filename"].tolist()
print("Running geometry query on parts:", len(urls))
if len(urls) == 0:
    raise RuntimeError("No candidate parts intersect this bbox (unexpected).")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("PRAGMA enable_progress_bar=false;")
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA temp_directory='/tmp';")

url_list_sql = ",\n".join([f"'{u}'" for u in urls])

sql = f"""
SELECT
  id,
  subtype,
  class,
  height,
  num_floors
FROM read_parquet([{url_list_sql}])
WHERE ST_Intersects(
  geometry,
  ST_MakeEnvelope({minx},{miny},{maxx},{maxy})
)
LIMIT 200
"""
df = con.execute(sql).df()
print("Matches returned:", len(df))
df.head(50)

Running geometry query on parts: 2
Matches returned: 200


,id,subtype,class,height,num_floors
0,68d961df-a34d-4398-994e-c0c980225336,residential,house,4.600000,<NA>
1,d612874b-b2ad-4567-9856-93b3664057b8,residential,static_caravan,4.600000,<NA>
2,c0724450-1e4e-48a1-b9ed-8d3dde92c909,residential,static_caravan,4.600000,<NA>
3,945314c5-9a78-414e-9374-ff033aae15cb,commercial,commercial,4.600000,<NA>
4,ad07c17e-6d78-4415-a290-788b9cf63b2e,residential,house,4.600000,<NA>
5,42bbc914-4bbf-4c46-86ba-9c4092d5594d,residential,house,4.600000,<NA>
6,67bbda89-4355-418c-96ec-065a9b7b2e2d,residential,house,4.600000,<NA>
7,54d847fb-4488-4bf9-99d4-b77989291c80,residential,house,4.600000,<NA>
8,b35363ee-6462-4c07-b3b0-ea891299127c,residential,house,4.600000,<NA>
9,9fba5d4a-6542-4950-96bf-cccf9714cd26,residential,house,4.600000,<NA>


In [8]:
import pandas as pd

# Amman bbox (Jordan) — adjust anytime
minx, miny, maxx, maxy = 35.80, 31.86, 36.12, 32.10

part_bbox = pd.read_parquet(REG / "overture_buildings_part_bbox.parquet")

cand = part_bbox[
    (part_bbox["xmax"] >= minx) &
    (part_bbox["xmin"] <= maxx) &
    (part_bbox["ymax"] >= miny) &
    (part_bbox["ymin"] <= maxy)
].copy()

print("Candidate parts for Amman bbox:", len(cand))
cand.sort_values(["xmin","ymin"]).head(25)

Candidate parts for Amman bbox: 2


,filename,xmin,ymin,xmax,ymax
156,https://overturemaps-us-west-2.s3.us-west-2.am...,22.499939,22.499983,45.000031,32.344704
31,https://overturemaps-us-west-2.s3.us-west-2.am...,34.852772,30.937338,45.000072,39.375084


In [9]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL spatial; LOAD spatial;")

# Amman bbox again
minx, miny, maxx, maxy = 35.80, 31.86, 36.12, 32.10

urls = cand["filename"].tolist()
print("Querying parts:", len(urls))

if len(urls) == 0:
    raise RuntimeError("No candidate parts found for this bbox. Expand bbox or confirm index file.")

url_list_sql = ",\n".join([f"'{u}'" for u in urls])

sql = f"""
SELECT
  id,
  subtype,
  class,
  height,
  num_floors,
  ST_AsText(ST_Centroid(geometry)) AS centroid_wkt
FROM read_parquet([{url_list_sql}])
WHERE ST_Intersects(
  geometry,
  ST_MakeEnvelope({minx},{miny},{maxx},{maxy})
)
LIMIT 200;
"""

df = con.execute(sql).df()
df

Querying parts: 2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,subtype,class,height,num_floors,centroid_wkt
0,17a30f0e-ef69-4708-97fb-79e0a2a67d81,None,None,NaN,<NA>,POINT (36.1177194 32.098997100000005)
1,ebebdd16-64ad-47f7-bc09-c21f8072ba68,None,None,NaN,<NA>,POINT (36.11742822560997 32.09887781193231)
2,4e9de578-51ab-4369-ad76-42734b9dce55,None,None,NaN,<NA>,POINT (36.11745631041721 32.098713479566285)
3,097a4569-a856-4473-960a-4011bfc34888,None,None,NaN,<NA>,POINT (36.114868799999996 32.09916155)
4,226c811a-1d88-46cc-98f8-f0ed620d04f8,None,None,NaN,<NA>,POINT (36.117200496754755 32.098300809081884)
...,...,...,...,...,...,...
195,c97812a5-e6fa-4306-b161-75e10575015c,None,None,NaN,<NA>,POINT (36.10506777206647 32.09716545482759)
196,da1657d2-b247-4ab2-97aa-daa71fd81cf3,None,None,NaN,<NA>,POINT (36.10426676748811 32.097184221082614)
197,f2703c3c-556f-4aae-94f0-f186f16f64c5,None,None,NaN,<NA>,POINT (36.10507596914218 32.097430161331395)
198,6afa3a10-7aea-4ca2-99c0-5d38bcc89a7a,None,None,NaN,<NA>,POINT (36.10460568337532 32.097349512958075)


In [10]:
print("Rows returned:", len(df))
df[["subtype","class"]].value_counts().head(20)

Rows returned: 200


subtype        class     
entertainment  grandstand    1
               stadium       1
Name: count, dtype: int64

In [15]:
# ============================================================
# FIX — robust URL joining (kills /./ and other path weirdness)
# ============================================================
from urllib.parse import urljoin, urlparse, urlunparse

def norm_url(u: str) -> str:
    """
    Normalize a URL path to remove '/./' and '/../' segments.
    urljoin handles most, but this ensures cleanup.
    """
    p = urlparse(u)
    # urljoin against itself will collapse dot segments in many cases
    cleaned = urljoin(u, p.path)
    # preserve query/fragment if any
    cp = urlparse(cleaned)
    return urlunparse((cp.scheme, cp.netloc, cp.path, p.params, p.query, p.fragment))

def join_url(base: str, href: str) -> str:
    return norm_url(urljoin(base, href))

In [16]:
# ============================================================
# CELL P1 (redo) — Find "places" theme catalog from release
# ============================================================
root = fetch_json(OVERTURE_ROOT)
release_url = f"https://stac.overturemaps.org/{LATEST_RELEASE}/catalog.json"
rel = fetch_json(release_url)

print("Root:", OVERTURE_ROOT)
print("Release catalog:", release_url)

places_href = None
for l in rel.get("links", []):
    if l.get("rel") == "child" and l.get("title") == "places":
        places_href = l.get("href")
        break

if not places_href:
    for l in rel.get("links", []):
        if l.get("rel") == "child" and "places" in (l.get("href","")):
            places_href = l.get("href")
            break

if not places_href:
    raise RuntimeError("Could not find 'places' in release catalog links.")

PLACES_CATALOG_URL = join_url(release_url, places_href)
print("Places catalog:", PLACES_CATALOG_URL)

# quick smoke
_ = fetch_json(PLACES_CATALOG_URL)
print("OK: fetched places catalog")

Root: https://stac.overturemaps.org/catalog.json
Release catalog: https://stac.overturemaps.org/2026-02-18.0/catalog.json
Places catalog: https://stac.overturemaps.org/2026-02-18.0/places/catalog.json
OK: fetched places catalog


In [17]:
# ============================================================
# CELL P2 — Inspect places catalog children (collections)
# ============================================================
places_cat = fetch_json(PLACES_CATALOG_URL)

rows = []
for l in places_cat.get("links", []):
    if l.get("rel") in ("child", "collection"):
        rows.append({
            "rel": l.get("rel"),
            "title": l.get("title"),
            "href": join_url(PLACES_CATALOG_URL, l.get("href","")),
            "type": l.get("type"),
        })

cdf = pd.DataFrame(rows)
print("Child/collection links:", len(cdf))
display(cdf.head(50))

Child/collection links: 1


,rel,title,href,type
0,child,None,https://stac.overturemaps.org/2026-02-18.0/pla...,application/json


In [18]:
# ============================================================
# CELL P3 — Follow the places child and list collections
# ============================================================

places_child_href = cdf.loc[0, "href"]
print("Places child:", places_child_href)

places_child = fetch_json(places_child_href)
print("type:", places_child.get("type"), "| id:", places_child.get("id"), "| title:", places_child.get("title"))

rows = []
for l in places_child.get("links", []):
    if l.get("rel") in ("child", "collection"):
        rows.append({
            "rel": l.get("rel"),
            "title": l.get("title"),
            "href": join_url(places_child_href, l.get("href","")),
            "type": l.get("type"),
        })

pcdf = pd.DataFrame(rows)
print("\nChild/collection links inside places child:", len(pcdf))
display(pcdf)

Places child: https://stac.overturemaps.org/2026-02-18.0/places/place/collection.json
type: Collection | id: place | title: None

Child/collection links inside places child: 0


""


In [19]:
# ============================================================
# CELL P4 — Inspect the places 'place' collection (assets + link stats)
# ============================================================

PLACE_COLLECTION_URL = places_child_href  # from P3
col = fetch_json(PLACE_COLLECTION_URL)

print("Opened:", PLACE_COLLECTION_URL)
print("type:", col.get("type"), "| id:", col.get("id"), "| title:", col.get("title"))
print("description:", (col.get("description") or "")[:120], "...")

links = col.get("links", [])
assets = col.get("assets", {}) or {}

# Link stats
rels = pd.Series([l.get("rel") for l in links]).value_counts(dropna=False)
print("\nTop rel counts:")
print(rels.head(15))

n_item_links = sum(1 for l in links if l.get("rel") == "item")
n_items_link = sum(1 for l in links if l.get("rel") == "items")
print("\nCounts:")
print("item links:", n_item_links)
print("items links:", n_items_link)
print("assets:", len(assets))

# Assets table
arows = []
for k, a in assets.items():
    arows.append({
        "asset_key": k,
        "href": a.get("href"),
        "type": a.get("type"),
        "roles": ",".join(a.get("roles", []) or []),
        "title": a.get("title"),
    })

adf = pd.DataFrame(arows)
if len(adf):
    adf["is_parquet"] = (
        adf["href"].fillna("").str.contains(r"\.parquet(\?|$)", regex=True) |
        adf["type"].fillna("").str.contains("parquet", case=False)
    )
    display(adf.sort_values(["is_parquet", "asset_key"], ascending=[False, True]))
else:
    print("\nNo assets found on this collection.")

Opened: https://stac.overturemaps.org/2026-02-18.0/places/place/collection.json
type: Collection | id: place | title: None
description: Overture's place collection ...

Top rel counts:
item      8
root      1
parent    1
Name: count, dtype: int64

Counts:
item links: 8
items links: 0
assets: 0

No assets found on this collection.


In [20]:
# ============================================================
# CELL P5 — Build Overture Places (place) files index from item JSONs
#   - Uses item.bbox for spatial coverage (FAST)
#   - Extracts parquet asset URLs from each item (aws/azure if present)
# ============================================================

import json
import urllib.request
from urllib.parse import urljoin, urlparse
import pandas as pd

def fetch_json(url: str) -> dict:
    with urllib.request.urlopen(url) as f:
        return json.load(f)

def join_url(base: str, href: str) -> str:
    # handles ./ and relative refs correctly
    return urljoin(base, href)

def guess_storage(href: str) -> str:
    h = (href or "").lower()
    if "blob.core.windows.net" in h:
        return "azure"
    if "s3" in h or "amazonaws.com" in h:
        return "aws"
    return "other"

REG = REG  # assumes already defined in your notebook

PLACE_COLLECTION_URL = "https://stac.overturemaps.org/2026-02-18.0/places/place/collection.json"
col = fetch_json(PLACE_COLLECTION_URL)

# Collect item URLs from collection links
item_links = [l for l in col.get("links", []) if l.get("rel") == "item" and l.get("href")]
item_urls = [join_url(PLACE_COLLECTION_URL, l["href"]) for l in item_links]

print("Collection:", PLACE_COLLECTION_URL)
print("Item links found:", len(item_urls))
print("First item URL:", item_urls[0] if item_urls else None)

rows = []
for item_url in item_urls:
    it = fetch_json(item_url)

    item_id = it.get("id")
    bbox = it.get("bbox") or [None, None, None, None]
    if len(bbox) != 4:
        bbox = [None, None, None, None]

    assets = it.get("assets", {}) or {}

    # keep only parquet-ish assets
    for asset_key, a in assets.items():
        href = a.get("href")
        a_type = a.get("type") or ""
        if not href:
            continue

        is_parquet = (".parquet" in href.lower()) or ("parquet" in a_type.lower())
        if not is_parquet:
            continue

        full_href = join_url(item_url, href)

        rows.append({
            "source_name": "overture",
            "release": "2026-02-18.0",
            "theme": "places",
            "layer": "place",
            "item_id": item_id,
            "asset_key": asset_key,
            "storage": guess_storage(full_href),
            "href": full_href,
            "type": a_type,
            "xmin": bbox[0],
            "ymin": bbox[1],
            "xmax": bbox[2],
            "ymax": bbox[3],
            "item_url": item_url,
        })

files = pd.DataFrame(rows)

if len(files) == 0:
    print("\nNo parquet assets found in item JSONs.")
else:
    out_path = REG / "overture_places_files.parquet"
    files.to_parquet(out_path, index=False)
    print(f"\nWROTE: {out_path} rows={len(files)}")

    print("\nCounts by storage:")
    display(files["storage"].value_counts(dropna=False))

    print("\nPreview:")
    display(files.head(20))

Collection: https://stac.overturemaps.org/2026-02-18.0/places/place/collection.json
Item links found: 8
First item URL: https://stac.overturemaps.org/2026-02-18.0/places/place/00000/00000.json

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_places_files.parquet rows=16

Counts by storage:


storage
aws      8
azure    8
Name: count, dtype: int64


Preview:


,source_name,release,theme,layer,item_id,asset_key,storage,href,type,xmin,ymin,xmax,ymax,item_url
0,overture,2026-02-18.0,places,place,00000,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-1.799980e+02,-88.049360,-0.000022,22.499991,https://stac.overturemaps.org/2026-02-18.0/pla...
1,overture,2026-02-18.0,places,place,00000,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-1.799980e+02,-88.049360,-0.000022,22.499991,https://stac.overturemaps.org/2026-02-18.0/pla...
2,overture,2026-02-18.0,places,place,00001,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-1.800000e+02,11.369039,-56.271973,90.000000,https://stac.overturemaps.org/2026-02-18.0/pla...
3,overture,2026-02-18.0,places,place,00001,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-1.800000e+02,11.369039,-56.271973,90.000000,https://stac.overturemaps.org/2026-02-18.0/pla...
4,overture,2026-02-18.0,places,place,00002,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-9.000000e+01,22.505557,-0.000001,90.000000,https://stac.overturemaps.org/2026-02-18.0/pla...
5,overture,2026-02-18.0,places,place,00002,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-9.000000e+01,22.505557,-0.000001,90.000000,https://stac.overturemaps.org/2026-02-18.0/pla...
6,overture,2026-02-18.0,places,place,00003,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-9.000001e+01,0.000001,89.999435,44.999972,https://stac.overturemaps.org/2026-02-18.0/pla...
7,overture,2026-02-18.0,places,place,00003,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-9.000001e+01,0.000001,89.999435,44.999972,https://stac.overturemaps.org/2026-02-18.0/pla...
8,overture,2026-02-18.0,places,place,00004,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,4.241981e-06,11.251796,89.999989,44.999999,https://stac.overturemaps.org/2026-02-18.0/pla...
9,overture,2026-02-18.0,places,place,00004,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,4.241981e-06,11.251796,89.999989,44.999999,https://stac.overturemaps.org/2026-02-18.0/pla...


In [21]:
# ============================================================
# CELL P6 — bbox prefilter then query only matching parquet parts
# ============================================================

import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

files = pd.read_parquet(REG / "overture_places_files.parquet")

# Pick one storage to query (aws is usually easiest)
files_aws = files[files["storage"] == "aws"].copy()

# --- set bbox here (example: Amman-ish) ---
xmin, ymin, xmax, ymax = 35.85, 31.85, 36.10, 32.10

cand = files_aws[
    (files_aws["xmax"] >= xmin) &
    (files_aws["xmin"] <= xmax) &
    (files_aws["ymax"] >= ymin) &
    (files_aws["ymin"] <= ymax)
].copy()

print("Candidate parquet parts:", len(cand))
display(cand[["item_id","xmin","ymin","xmax","ymax","href"]].head(20))

if len(cand) == 0:
    raise RuntimeError("No candidate parts found for that bbox. Expand bbox or verify item bboxes.")

# Query ONLY candidate parquet files
urls = cand["href"].tolist()

# DuckDB: query a list of parquet files
df = con.execute("""
    SELECT
      id,
      names,
      confidence,
      categories,
      theme,
      type
    FROM read_parquet($1)
    WHERE ST_Intersects(
      geometry,
      ST_MakeEnvelope($2, $3, $4, $5)
    )
    LIMIT 200
""", [urls, xmin, ymin, xmax, ymax]).df()

df

Candidate parquet parts: 3


,item_id,xmin,ymin,xmax,ymax,href
6,00003,-90.000006,0.000001,89.999435,44.999972,https://overturemaps-us-west-2.s3.us-west-2.am...
8,00004,0.000004,11.251796,89.999989,44.999999,https://overturemaps-us-west-2.s3.us-west-2.am...
12,00006,0.065918,11.250000,180.000000,90.000000,https://overturemaps-us-west-2.s3.us-west-2.am...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,names,confidence,categories,theme,type
0,2bf67b51-6b59-4e11-a333-88732fb46ab0,"{'primary': 'مياه الدهمس', 'common': None, 'ru...",0.950063,{'primary': 'water_treatment_equipment_and_ser...,places,place
1,ad0d407e-9f2c-4b83-81bb-edf9fee16b8f,"{'primary': 'ابو مريم منتجات ألبان_اسواق مريم""...",0.323479,"{'primary': 'shopping', 'alternate': None}",places,place
2,f44c5e70-1af6-499b-b5ac-7479dbac970c,"{'primary': 'Ballerina', 'common': None, 'rule...",0.639236,"{'primary': 'shoe_store', 'alternate': ['retai...",places,place
3,86cf809a-5d64-4470-acdc-6aaf0f0b0b07,"{'primary': 'Raed Qandah Beauty Salon', 'commo...",0.950063,"{'primary': 'beauty_salon', 'alternate': ['bea...",places,place
4,92a96a31-4195-4413-9c5e-b25b15132db6,{'primary': 'Dr-ahmad badria الدكتور احمد بدر...,0.323479,"{'primary': 'doctor', 'alternate': ['coffee_sh...",places,place
...,...,...,...,...,...,...
195,fcd941cc-16f4-4a83-9e88-7081d0192ef5,"{'primary': 'Jordan Gate National Academy', 'c...",0.323479,"{'primary': 'college_university', 'alternate':...",places,place
196,e7e03bd2-f4aa-4a2f-b4d4-98c1d275ece7,{'primary': 'Leen Center for Translation مركز ...,0.639236,"{'primary': 'community_services_non_profits', ...",places,place
197,990a963a-a4b0-42f3-9eed-5040b44ff713,"{'primary': 'صيدلية بالي - Bali pharmacy', 'co...",0.639236,"{'primary': 'hospital', 'alternate': ['pharmac...",places,place
198,5745fa84-fa51-43b1-8e52-44cc5b226fc0,"{'primary': 'مؤسسة شاهين للمعدات الصناعية', 'c...",0.323479,"{'primary': 'building_supply_store', 'alternat...",places,place


In [ ]:
import pandas as pd

files = pd.read_parquet(REG / "overture_places_files.parquet")
print("items:", files["item_id"].nunique())
print("storages:", files["storage"].unique())

# how many items overlap Amman bbox?
xmin, ymin, xmax, ymax = 35.85, 31.85, 36.10, 32.10
cand = files[(files["storage"]=="aws") &
             (files["xmax"] >= xmin) &
             (files["xmin"] <= xmax) &
             (files["ymax"] >= ymin) &
             (files["ymin"] <= ymax)]
cand[["item_id","xmin","ymin","xmax","ymax"]]

In [22]:
import pandas as pd

places = pd.read_parquet(REG / "overture_places_files.parquet")
print("rows:", len(places))
print("aws parts:", len(places[places.storage=="aws"]))
print("bbox null rate:", places[["xmin","ymin","xmax","ymax"]].isna().mean())
places.head(3)[["item_id","storage","xmin","ymin","xmax","ymax"]]

rows: 16
aws parts: 8
bbox null rate: xmin    0.0
ymin    0.0
xmax    0.0
ymax    0.0
dtype: float64


,item_id,storage,xmin,ymin,xmax,ymax
0,00000,aws,-179.998035,-88.049360,-0.000022,22.499991
1,00000,azure,-179.998035,-88.049360,-0.000022,22.499991
2,00001,aws,-180.000000,11.369039,-56.271973,90.000000


In [23]:
# ============================================================
# CELL A1 — Addresses: helpers + resolve latest release + addresses catalog
# ============================================================

import json
import urllib.request
from urllib.parse import urljoin
import pandas as pd

def fetch_json(url: str) -> dict:
    with urllib.request.urlopen(url) as f:
        return json.load(f)

def abs_href(base_url: str, href: str) -> str:
    # Overture catalogs often return relative hrefs like "./2026-02-18.0/catalog.json"
    return urljoin(base_url, href)

ROOT = "https://stac.overturemaps.org/catalog.json"
root = fetch_json(ROOT)

# pick the "Latest Overture Release" child
latest = None
for l in root.get("links", []):
    if l.get("rel") == "child" and l.get("title") and "Latest" in l.get("title"):
        latest = l
        break

if latest is None:
    # fallback: take the first child link
    latest = next(l for l in root.get("links", []) if l.get("rel") == "child")

LATEST_RELEASE_HREF = abs_href(ROOT, latest["href"])
release_cat = fetch_json(LATEST_RELEASE_HREF)

# find "addresses" child in the release catalog
addresses_link = None
for l in release_cat.get("links", []):
    if l.get("rel") == "child" and (l.get("title") == "addresses" or "addresses" in (l.get("href") or "")):
        addresses_link = l
        break

if addresses_link is None:
    raise RuntimeError("Could not find 'addresses' child link in the latest release catalog.")

ADDRESSES_CATALOG_URL = abs_href(LATEST_RELEASE_HREF, addresses_link["href"])

print("Root:", ROOT)
print("Latest release catalog:", LATEST_RELEASE_HREF)
print("Addresses catalog:", ADDRESSES_CATALOG_URL)

Root: https://stac.overturemaps.org/catalog.json
Latest release catalog: https://stac.overturemaps.org/2026-02-18.0/catalog.json
Addresses catalog: https://stac.overturemaps.org/2026-02-18.0/addresses/catalog.json


In [24]:
# ============================================================
# CELL A2 — Addresses: inspect addresses catalog links
# ============================================================

addresses_cat = fetch_json(ADDRESSES_CATALOG_URL)

rows = []
for l in addresses_cat.get("links", []):
    if l.get("rel") in ("child", "collection"):
        rows.append({
            "rel": l.get("rel"),
            "title": l.get("title"),
            "href": abs_href(ADDRESSES_CATALOG_URL, l.get("href","")),
            "type": l.get("type")
        })

df_links = pd.DataFrame(rows)
print("Child/collection links:", len(df_links))
display(df_links)

Child/collection links: 1


,rel,title,href,type
0,child,None,https://stac.overturemaps.org/2026-02-18.0/add...,application/json


In [25]:
# ============================================================
# CELL A3 — Addresses: resolve addresses collection.json
# ============================================================

# Prefer a direct collection.json link if present.
collection_url = None
for _, r in df_links.iterrows():
    if isinstance(r.get("href"), str) and r["href"].endswith("collection.json"):
        collection_url = r["href"]
        break

# Otherwise, open the child catalog and find its collection.json
if collection_url is None:
    if len(df_links) == 0:
        raise RuntimeError("No child/collection links found in addresses catalog.")
    child_url = df_links.iloc[0]["href"]
    child = fetch_json(child_url)

    # Sometimes the child is already a Collection; handle both
    if child.get("type") == "Collection":
        collection_url = child_url
    else:
        # child is a Catalog -> find collection.json in its links
        for l in child.get("links", []):
            if l.get("rel") in ("child","collection") and (l.get("href","").endswith("collection.json")):
                collection_url = abs_href(child_url, l["href"])
                break

if collection_url is None:
    raise RuntimeError("Could not resolve addresses collection.json URL.")

coll = fetch_json(collection_url)
print("Opened:", collection_url)
print("type:", coll.get("type"), "| id:", coll.get("id"), "| title:", coll.get("title"))
print("description:", (coll.get("description") or "")[:120], "...")

Opened: https://stac.overturemaps.org/2026-02-18.0/addresses/address/collection.json
type: Collection | id: address | title: None
description: Overture's address collection ...


In [26]:
# ============================================================
# CELL A4 — Addresses: collect item links
# ============================================================

item_links = [l for l in coll.get("links", []) if l.get("rel") == "item"]
print("Item links found:", len(item_links))

if len(item_links) == 0:
    # Some STACs use "items" or put item links elsewhere, but Overture typically uses rel=item
    rel_counts = pd.Series([l.get("rel") for l in coll.get("links", [])]).value_counts()
    print("Top rel counts:")
    display(rel_counts.head(15))
    raise RuntimeError("No rel=item links found in addresses collection. Need to inspect collection links.")

first_item_url = abs_href(collection_url, item_links[0]["href"])
print("First item URL:", first_item_url)

Item links found: 19
First item URL: https://stac.overturemaps.org/2026-02-18.0/addresses/address/00000/00000.json


In [27]:
# ============================================================
# CELL A5 — Addresses: extract parquet assets + bbox per item, write files table
# ============================================================

from hashlib import blake2b

def stable_id(*parts: str) -> str:
    h = blake2b(digest_size=8)
    for p in parts:
        h.update((p or "").encode("utf-8"))
        h.update(b"|")
    return h.hexdigest()

rows = []
for l in item_links:
    item_url = abs_href(collection_url, l["href"])
    item = fetch_json(item_url)

    bbox = item.get("bbox")  # [xmin, ymin, xmax, ymax]
    if not bbox or len(bbox) != 4:
        xmin = ymin = xmax = ymax = None
    else:
        xmin, ymin, xmax, ymax = bbox

    theme = item.get("properties", {}).get("theme")
    layer = item.get("properties", {}).get("type")
    item_id = item.get("id")

    for asset_key, asset in (item.get("assets") or {}).items():
        href = asset.get("href")
        typ  = asset.get("type")
        if not href:
            continue
        if "parquet" not in (typ or "").lower() and ".parquet" not in href.lower():
            continue

        storage = "azure" if "blob.core.windows.net" in href else ("aws" if "amazonaws.com" in href else "other")

        rows.append({
            "file_id": stable_id("overture", str(item_id), str(asset_key), str(href)),
            "source_name": "overture",
            "release": LATEST_RELEASE_HREF.split("/")[-2].replace("catalog.json","").strip("/"),  # best-effort
            "theme": theme or "addresses",
            "layer": layer or "address",
            "item_id": item_id,
            "asset_key": asset_key,
            "storage": storage,
            "href": href,
            "type": typ,
            "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
            "item_url": item_url,
        })

adf = pd.DataFrame(rows)

out_path = REG / "overture_addresses_files.parquet"
adf.to_parquet(out_path, index=False)

print("WROTE:", out_path, "rows=", len(adf))
print("\nCounts by storage:")
display(adf["storage"].value_counts())

display(adf.head(10))

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_addresses_files.parquet rows= 38

Counts by storage:


storage
aws      19
azure    19
Name: count, dtype: int64

,file_id,source_name,release,theme,layer,item_id,asset_key,storage,href,type,xmin,ymin,xmax,ymax,item_url
0,b4c5dce4b46d9a13,overture,2026-02-18.0,addresses,address,00000,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-73.847827,-54.938042,-45.000011,-22.500002,https://stac.overturemaps.org/2026-02-18.0/add...
1,0f3b533a428faebd,overture,2026-02-18.0,addresses,address,00000,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-73.847827,-54.938042,-45.000011,-22.500002,https://stac.overturemaps.org/2026-02-18.0/add...
2,10dc6ffe0510947c,overture,2026-02-18.0,addresses,address,00001,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-58.425471,-34.973329,-37.968754,-11.250001,https://stac.overturemaps.org/2026-02-18.0/add...
3,0a3b4e1bf36a2b21,overture,2026-02-18.0,addresses,address,00001,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-58.425471,-34.973329,-37.968754,-11.250001,https://stac.overturemaps.org/2026-02-18.0/add...
4,79b816dae372d6cb,overture,2026-02-18.0,addresses,address,00002,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-56.250000,-16.874393,-32.398186,-0.000069,https://stac.overturemaps.org/2026-02-18.0/add...
5,e1f63bf58d1b6a7a,overture,2026-02-18.0,addresses,address,00002,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-56.250000,-16.874393,-32.398186,-0.000069,https://stac.overturemaps.org/2026-02-18.0/add...
6,e03b9d908eabe0b8,overture,2026-02-18.0,addresses,address,00003,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-176.874527,-44.259077,-45.000004,19.687499,https://stac.overturemaps.org/2026-02-18.0/add...
7,02598475e8dc8cf0,overture,2026-02-18.0,addresses,address,00003,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-176.874527,-44.259077,-45.000004,19.687499,https://stac.overturemaps.org/2026-02-18.0/add...
8,76f4c32cadbf4d4a,overture,2026-02-18.0,addresses,address,00004,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-118.499924,16.874999,-90.000003,33.749996,https://stac.overturemaps.org/2026-02-18.0/add...
9,264334c1064a5db5,overture,2026-02-18.0,addresses,address,00004,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-118.499924,16.874999,-90.000003,33.749996,https://stac.overturemaps.org/2026-02-18.0/add...


In [61]:
# ============================================================
# CELL A6.U1 — Build Overture ADDRESSES grid index (REGENERATED)
#   Output: overture_addresses_grid_index.parquet
# ============================================================

from pathlib import Path
import pandas as pd
import duckdb
import numpy as np
from tqdm.auto import tqdm

OUT = REG / "overture_addresses_grid_index.parquet"

# ---- Inputs: you already created overture_addresses_files.parquet (like places/buildings did)
# It should contain at least: storage, href, xmin,ymin,xmax,ymax (and ideally item_id etc.)
addr_files = pd.read_parquet(REG / "overture_addresses_files.parquet")

# Use AWS only (Azure is duplicate content for our indexing purposes)
aws = addr_files[addr_files["storage"] == "aws"].copy()
aws = aws.drop_duplicates(subset=["href"]).reset_index(drop=True)

print("AWS parts:", len(aws))

# ---- Grid resolution
CELL_DEG = 0.25  # keep as you chose
LON0 = -180.0
LAT0 = -90.0

def cell_bbox_from_bins(lon_bin: int, lat_bin: int, cell_deg: float):
    xmin = LON0 + lon_bin * cell_deg
    xmax = xmin + cell_deg
    ymin = LAT0 + lat_bin * cell_deg
    ymax = ymin + cell_deg
    return xmin, ymin, xmax, ymax

def make_cell_id(lon_bin: int, lat_bin: int, cell_deg: float):
    return f"g{cell_deg}_{lon_bin}_{lat_bin}"

# ---- DuckDB connection (spatial not required for bbox binning)
con = duckdb.connect()
con.execute("PRAGMA threads=4;")  # adjust if you want

# ---- Scan strategy
# For addresses, full scans can be huge. Start with SAMPLE_N to validate workflow.
# Set SAMPLE_N = None to scan full files (slowest, biggest).
SAMPLE_N = None   # try 2M rows per part; bump later (or None for full scan)

rows_out = []

for href in tqdm(aws["href"].tolist(), desc="Indexing parts"):
    # Build a safe query with no fragile CTE formatting.
    # We read bbox struct, filter nulls, compute bins, then group.
    if SAMPLE_N is None:
        scan_clause = f"FROM read_parquet('{href}')"
    else:
        # USING SAMPLE avoids pulling all rows into memory; still scans file but limits output.
        scan_clause = f"FROM (SELECT bbox FROM read_parquet('{href}') USING SAMPLE {int(SAMPLE_N)} ROWS)"

    q = f"""
    SELECT
      CAST(FLOOR((bbox.xmin - ({LON0})) / {CELL_DEG}) AS BIGINT) AS lon_bin,
      CAST(FLOOR((bbox.ymin - ({LAT0})) / {CELL_DEG}) AS BIGINT) AS lat_bin,
      COUNT(*) AS n_in_cell
    {scan_clause}
    WHERE
      bbox IS NOT NULL
      AND bbox.xmin IS NOT NULL AND bbox.ymin IS NOT NULL
    GROUP BY lon_bin, lat_bin
    """

    df = con.execute(q).df()
    if df.empty:
        continue

    df["href"] = href
    df["cell_deg"] = CELL_DEG
    df["cell_id"] = [make_cell_id(int(a), int(b), CELL_DEG) for a, b in zip(df["lon_bin"], df["lat_bin"])]

    # attach cell bbox now (cheap + deterministic)
    bb = np.array([cell_bbox_from_bins(int(a), int(b), CELL_DEG) for a, b in zip(df["lon_bin"], df["lat_bin"])])
    df["cell_xmin"] = bb[:, 0]
    df["cell_ymin"] = bb[:, 1]
    df["cell_xmax"] = bb[:, 2]
    df["cell_ymax"] = bb[:, 3]

    rows_out.append(df)

grid = pd.concat(rows_out, ignore_index=True) if rows_out else pd.DataFrame()

# ---- Optional: compute dominant country per cell (approx)
# This requires country column in the parquet parts. If your address files have it, we can compute mode.
# We'll do it in U2 (cleaner), since U1 should just build the part->cell index.

# ---- Add identifiers
grid["release"] = "2026-02-18.0"
grid["theme"] = "addresses"
grid["layer"] = "address"
grid["storage"] = "aws"

# ---- Column order
cols = [
    "release","theme","layer","storage",
    "href",
    "cell_id","cell_deg","lon_bin","lat_bin",
    "cell_xmin","cell_ymin","cell_xmax","cell_ymax",
    "n_in_cell"
]
grid = grid[cols].sort_values(["href","lon_bin","lat_bin"]).reset_index(drop=True)

grid.to_parquet(OUT, index=False)
print("WROTE:", OUT, "rows=", len(grid))
grid.head(10)

AWS parts: 19


Indexing parts:   0%|          | 0/19 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_addresses_grid_index.parquet rows= 43668


,release,theme,layer,storage,href,cell_id,cell_deg,lon_bin,lat_bin,cell_xmin,cell_ymin,cell_xmax,cell_ymax,n_in_cell
0,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_424_189,0.25,424,189,-74.00,-42.75,-73.75,-42.50,1539
1,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_424_190,0.25,424,190,-74.00,-42.50,-73.75,-42.25,10832
2,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_424_192,0.25,424,192,-74.00,-42.00,-73.75,-41.75,10271
3,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_424_197,0.25,424,197,-74.00,-40.75,-73.75,-40.50,2
4,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_425_187,0.25,425,187,-73.75,-43.25,-73.50,-43.00,3424
5,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_425_190,0.25,425,190,-73.75,-42.50,-73.50,-42.25,2894
6,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_425_193,0.25,425,193,-73.75,-41.75,-73.50,-41.50,734
7,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_425_197,0.25,425,197,-73.75,-40.75,-73.50,-40.50,1245
8,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_425_209,0.25,425,209,-73.75,-37.75,-73.50,-37.50,7371
9,2026-02-18.0,addresses,address,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_426_188,0.25,426,188,-73.50,-43.00,-73.25,-42.75,104


In [63]:
# ============================================================
# CELL A6.U2 — Addresses: add country_mode per cell (FULL SCAN if SAMPLE_N=None)
# ============================================================

from pathlib import Path
import duckdb
import pandas as pd
from tqdm.auto import tqdm

# Inputs
GRID_PATH = REG / "overture_addresses_grid_index.parquet"   # output of A6.U1
OUT_PATH  = REG / "overture_addresses_grid_cells.parquet"   # new: cell summary (country_mode, counts)
CELL_DEG  = 0.25

# IMPORTANT:
# - If SAMPLE_N is None => full scan per part (safe but slower)
# - If SAMPLE_N is int  => sample per part (faster but may miss rare cells)
SAMPLE_N = None   # set to an int like 2_000_000 if you want sampling

grid = pd.read_parquet(GRID_PATH)

# We'll compute country_mode at the cell level by aggregating per (lon_bin, lat_bin, country)
hrefs = sorted(grid.loc[grid["storage"].eq("aws"), "href"].unique())
print("Parts (aws):", len(hrefs))
print("Grid rows:", len(grid), "| unique cells:", grid["cell_id"].nunique())

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

def sample_sql_clause(sample_n):
    """Return DuckDB SQL for optional sampling."""
    if sample_n is None:
        return ""
    if isinstance(sample_n, int) and sample_n > 0:
        return f" USING SAMPLE {sample_n} ROWS"
    raise ValueError(f"SAMPLE_N must be None or positive int, got: {sample_n!r}")

SAMPLE_CLAUSE = sample_sql_clause(SAMPLE_N)

pieces = []

for href in tqdm(hrefs, desc="Country mode per part"):
    q = f"""
    SELECT
      CAST(FLOOR((bbox.xmin - (-180.0)) / {CELL_DEG}) AS BIGINT) AS lon_bin,
      CAST(FLOOR((bbox.ymin - (-90.0))  / {CELL_DEG}) AS BIGINT) AS lat_bin,
      country,
      COUNT(*) AS n
    FROM (
      SELECT bbox, country
      FROM read_parquet('{href}'){SAMPLE_CLAUSE}
    )
    WHERE bbox IS NOT NULL
      AND bbox.xmin IS NOT NULL AND bbox.ymin IS NOT NULL
      AND country IS NOT NULL AND country <> ''
    GROUP BY lon_bin, lat_bin, country
    """
    df = con.execute(q).df()
    if not df.empty:
        pieces.append(df)

if not pieces:
    raise RuntimeError("No country rows produced. Something is wrong with bbox/country fields.")

country_counts = pd.concat(pieces, ignore_index=True)

# Aggregate across parts (some cells may appear in >1 part)
country_counts = (
    country_counts
    .groupby(["lon_bin", "lat_bin", "country"], as_index=False)["n"].sum()
)

# Pick mode per cell
country_mode = (
    country_counts
    .sort_values(["lon_bin", "lat_bin", "n", "country"], ascending=[True, True, False, True])
    .groupby(["lon_bin", "lat_bin"], as_index=False)
    .first()
    .rename(columns={"country": "country_mode", "n": "n_country_mode"})
)

# Also total counts per cell (from grid’s n_in_cell, which is already part-level aggregated)
cell_counts = (
    grid.groupby(["lon_bin", "lat_bin"], as_index=False)["n_in_cell"].sum()
    .rename(columns={"n_in_cell": "n_in_cell_total"})
)

cells = cell_counts.merge(country_mode, on=["lon_bin", "lat_bin"], how="left")

# Build cell_id + bbox from bins
cells["cell_deg"] = CELL_DEG
cells["cell_id"] = "g0.25_" + cells["lon_bin"].astype(str) + "_" + cells["lat_bin"].astype(str)

cells["xmin"] = -180.0 + cells["lon_bin"] * CELL_DEG
cells["ymin"] =  -90.0 + cells["lat_bin"] * CELL_DEG
cells["xmax"] = cells["xmin"] + CELL_DEG
cells["ymax"] = cells["ymin"] + CELL_DEG

# Order columns
cells = cells[
    ["cell_id", "cell_deg", "lon_bin", "lat_bin", "xmin", "ymin", "xmax", "ymax",
     "n_in_cell_total", "country_mode", "n_country_mode"]
].sort_values(["lon_bin", "lat_bin"]).reset_index(drop=True)

cells.to_parquet(OUT_PATH, index=False)
print("WROTE:", OUT_PATH, "rows=", len(cells))
display(cells.head(20))

Parts (aws): 19
Grid rows: 43668 | unique cells: 42789


Country mode per part:   0%|          | 0/19 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_addresses_grid_cells.parquet rows= 42789


,cell_id,cell_deg,lon_bin,lat_bin,xmin,ymin,xmax,ymax,n_in_cell_total,country_mode,n_country_mode
0,g0.25_12_184,0.25,12,184,-177.00,-44.00,-176.75,-43.75,5,NZ,5
1,g0.25_13_183,0.25,13,183,-176.75,-44.25,-176.50,-44.00,6,NZ,6
2,g0.25_13_184,0.25,13,184,-176.75,-44.00,-176.50,-43.75,259,NZ,259
3,g0.25_13_185,0.25,13,185,-176.75,-43.75,-176.50,-43.50,5,NZ,5
4,g0.25_14_183,0.25,14,183,-176.50,-44.25,-176.25,-44.00,39,NZ,39
5,g0.25_14_184,0.25,14,184,-176.50,-44.00,-176.25,-43.75,20,NZ,20
6,g0.25_14_185,0.25,14,185,-176.50,-43.75,-176.25,-43.50,56,NZ,56
7,g0.25_15_182,0.25,15,182,-176.25,-44.50,-176.00,-44.25,1,NZ,1
8,g0.25_15_183,0.25,15,183,-176.25,-44.25,-176.00,-44.00,1,NZ,1
9,g0.25_15_185,0.25,15,185,-176.25,-43.75,-176.00,-43.50,4,NZ,4


In [64]:
# ============================================================
# CELL A7.S2 — Smarter query: pick smallest candidates first
# ============================================================

import pandas as pd
import duckdb, math
from pathlib import Path

REG = Path("/Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry")
idx = pd.read_parquet(REG / "overture_addresses_grid_index.parquet")

CELL_DEG = 0.25

def bbox_to_grid_cells(bbox, cell_deg):
    minx, miny, maxx, maxy = bbox
    lon0 = math.floor((minx + 180.0) / cell_deg)
    lon1 = math.floor((maxx + 180.0) / cell_deg)
    lat0 = math.floor((miny +  90.0) / cell_deg)
    lat1 = math.floor((maxy +  90.0) / cell_deg)
    return [f"g{cell_deg}_{xb}_{yb}" for xb in range(lon0, lon1+1) for yb in range(lat0, lat1+1)]

BBOXES = {
    "san_diego": (-117.35, 32.6, -116.9, 33.1),
    "berlin": (13.0884, 52.3383, 13.7611, 52.6755),
    "amman": (35.7, 31.85, 36.1, 32.1),
    "riyadh": (46.52, 24.52, 47.02, 24.92),
}

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("SET threads=8;")
con.execute("SET memory_limit='6GB';")
con.execute(f"SET temp_directory='{REG / '_duckdb_tmp'}';")

for city, bbox in BBOXES.items():
    minx, miny, maxx, maxy = bbox
    cells = bbox_to_grid_cells(bbox, CELL_DEG)

    # narrow by cells, then rank candidate parts by total rows they contribute in those cells
    cand = idx[idx["cell_id"].isin(cells)].copy()
    if cand.empty:
        print("\nCITY:", city, "=> no candidate parts")
        continue

    part_rank = (cand.groupby("href")["n_in_cell"].sum()
                 .sort_values()
                 .reset_index()
                 .rename(columns={"n_in_cell": "rank_rows"}))

    # grab the smallest N parts first
    TOP_N = 8
    hrefs = part_rank["href"].head(TOP_N).tolist()

    print("\n==============================")
    print("CITY:", city, "| grid cells:", len(cells), "| candidate parts:", cand['href'].nunique(), "| using:", len(hrefs))

    union = " UNION ALL ".join([f"SELECT * FROM read_parquet('{h}')" for h in hrefs])
    q = f"""
    SELECT id, country, postcode, street, number, postal_city
    FROM ({union})
    WHERE bbox.xmin <= {maxx} AND bbox.xmax >= {minx}
      AND bbox.ymin <= {maxy} AND bbox.ymax >= {miny}
    LIMIT 50
    """
    res = con.execute(q).df()
    print("rows:", len(res))
    display(res.head(50))


CITY: san_diego | grid cells: 9 | candidate parts: 1 | using: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 50


,id,country,postcode,street,number,postal_city
0,a6524077-816b-4e9a-b310-808f4ba029f2,US,91911,AVENIDA ROSA,1518,None
1,633f2de2-d64e-4f47-af72-9ac6885237cf,US,91911,03RD AVE,1500,None
2,b67ae3c0-9ce9-468d-853a-3c47f4491ad9,US,91911,03RD AVE,1500,None
3,ddb70193-ea5f-4e36-9b46-0ceda5d91bae,US,91911,AVENIDA ROSA,1504,None
4,3c40a5b0-453b-4634-9b8f-d8168250cf36,US,91911,AVENIDA ROSA,1506,None
5,3f53dacd-599a-4f23-8e05-cc9ff02fd926,US,91911,AVENIDA ROSA,1516,None
6,a57868f5-1d66-4f7c-a468-16d3b10bcd33,US,91911,AVENIDA ROSA,1512,None
7,a1e719ee-464b-466b-b377-a9be41385a0c,US,91911,AVENIDA ROSA,1510,None
8,217b9fe7-de19-4144-a157-94348c5f8f17,US,91911,AVENIDA ROSA,1500,None
9,f99795c7-3448-4abf-8479-eceabc416652,US,91911,AVENIDA ROSA,1505,None



CITY: berlin | grid cells: 8 | candidate parts: 1 | using: 1
rows: 50


,id,country,postcode,street,number,postal_city
0,d04f7dd4-fbbf-452e-9e57-009616eded31,DE,15345,Alte Schulstraße,1a,None
1,e355bab5-7ae7-4b81-94c0-5d733deb4e6c,DE,15345,Alte Poststraße,25,None
2,fda14748-80d3-4ce8-896c-0674a7ca953f,DE,15345,Alte Schulstraße,44,None
3,24183d97-8f84-49ea-be19-e373cd4fc8e6,DE,15345,Alte Poststraße,1,None
4,bb0775db-cddb-479e-bfd7-8a5b04cdeafd,DE,15345,Alte Poststraße,1a,None
5,9ce2defb-a8d2-4719-94f2-6040ffc31563,DE,15345,Alte Poststraße,24,None
6,ac65f182-8cf3-4761-950e-a2f57c505003,DE,15345,Alte Poststraße,2a,None
7,be7b4949-2530-4f4e-98ef-e35087042108,DE,15345,Alte Poststraße,2,None
8,774cd7c8-9f14-4909-ba58-e94f7b6b2e7a,DE,15345,Alte Poststraße,2b,None
9,7553fed3-727a-483b-8f45-b18316bab6fc,DE,15345,Alte Poststraße,2c,None



CITY: amman => no candidate parts

CITY: riyadh => no candidate parts


In [65]:
# ============================================================
# CELL T1 — Locate transportation catalog + collections
# ============================================================

import json, urllib.request
import pandas as pd

ROOT = "https://stac.overturemaps.org/catalog.json"
RELEASE = "2026-02-18.0"  # keep explicit for now

def fetch_json(url: str) -> dict:
    with urllib.request.urlopen(url) as f:
        return json.load(f)

root = fetch_json(ROOT)
release_href = None
for l in root.get("links", []):
    if l.get("rel") in ("child","item") and str(l.get("href","")).endswith(f"{RELEASE}/catalog.json"):
        release_href = l["href"]
        break

if not release_href:
    # fallback: construct absolute
    release_href = f"https://stac.overturemaps.org/{RELEASE}/catalog.json"

release_url = release_href if release_href.startswith("http") else urllib.parse.urljoin(ROOT, release_href)
rel = fetch_json(release_url)

# find transportation child
transport_href = None
for l in rel.get("links", []):
    if l.get("rel") == "child" and str(l.get("href","")).rstrip("/").endswith("/transportation/catalog.json"):
        transport_href = l["href"]
        break

if not transport_href:
    # robust fallback: search any child containing '/transportation/'
    for l in rel.get("links", []):
        if l.get("rel") == "child" and "/transportation/" in str(l.get("href","")):
            transport_href = l["href"]; break

if not transport_href:
    raise RuntimeError("Could not find transportation catalog link in release catalog links.")

transport_url = transport_href if transport_href.startswith("http") else urllib.parse.urljoin(release_url, transport_href)

print("Root:", ROOT)
print("Release catalog:", release_url)
print("Transportation catalog:", transport_url)

transport_cat = fetch_json(transport_url)
links = []
for l in transport_cat.get("links", []):
    if l.get("rel") in ("child","collection") and l.get("type") == "application/json":
        links.append({"rel": l.get("rel"), "title": l.get("title"), "href": l.get("href"), "type": l.get("type")})

df = pd.DataFrame(links)
print("Child/collection links:", len(df))
df

Root: https://stac.overturemaps.org/catalog.json
Release catalog: https://stac.overturemaps.org/2026-02-18.0/catalog.json
Transportation catalog: https://stac.overturemaps.org/2026-02-18.0/transportation/catalog.json
Child/collection links: 2


,rel,title,href,type
0,child,None,./connector/collection.json,application/json
1,child,None,./segment/collection.json,application/json


In [66]:
# ============================================================
# CELL T2 — Inspect transportation collections
# ============================================================

import urllib.parse
import pandas as pd

collections = []
for _, r in df.iterrows():
    href = r["href"]
    url = href if href.startswith("http") else urllib.parse.urljoin(transport_url, href)
    js = fetch_json(url)

    links = js.get("links", [])
    rels = pd.Series([x.get("rel") for x in links]).value_counts()

    collections.append({
        "href": url,
        "type": js.get("type"),
        "id": js.get("id"),
        "title": js.get("title"),
        "n_links": len(links),
        "n_item_links": int(rels.get("item", 0)),
        "n_assets": len(js.get("assets", {}) or {}),
        "description": (js.get("description") or "")[:120]
    })

cdf = pd.DataFrame(collections).sort_values(["id","href"]).reset_index(drop=True)
cdf

,href,type,id,title,n_links,n_item_links,n_assets,description
0,https://stac.overturemaps.org/2026-02-18.0/tra...,Collection,connector,None,22,20,0,Overture's connector collection
1,https://stac.overturemaps.org/2026-02-18.0/tra...,Collection,segment,None,63,61,0,Overture's segment collection


In [67]:
# ============================================================
# CELL T3 — Build overture_transportation_files.parquet
# ============================================================

from pathlib import Path
import pandas as pd
import urllib.parse

REG = Path("/Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry")
OUT = REG / "overture_transportation_files.parquet"

rows = []

for _, c in cdf.iterrows():
    coll_url = c["href"]
    coll = fetch_json(coll_url)

    # item links are in collection.links rel=item
    item_links = [l for l in coll.get("links", []) if l.get("rel") == "item"]
    if not item_links:
        continue

    for l in item_links:
        item_url = l["href"]
        item_url = item_url if item_url.startswith("http") else urllib.parse.urljoin(coll_url, item_url)
        item = fetch_json(item_url)

        bbox = item.get("bbox", None)
        if bbox and len(bbox) >= 4:
            xmin, ymin, xmax, ymax = bbox[0], bbox[1], bbox[2], bbox[3]
        else:
            xmin = ymin = xmax = ymax = None

        assets = item.get("assets", {}) or {}
        for asset_key, a in assets.items():
            href = a.get("href")
            if not href:
                continue
            atype = (a.get("type") or "").lower()
            if ("parquet" not in atype) and (".parquet" not in href.lower()):
                continue

            storage = "aws" if "overturemaps-us-west-2" in href else ("azure" if "blob.core.windows.net" in href else "other")

            rows.append({
                "source_name": "overture",
                "release": RELEASE,
                "theme": "transportation",
                "layer": c["id"],
                "item_id": item.get("id"),
                "asset_key": asset_key,
                "storage": storage,
                "href": href,
                "type": a.get("type"),
                "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
                "item_url": item_url,
                "collection_url": coll_url
            })

files = pd.DataFrame(rows)

if files.empty:
    raise RuntimeError("No parquet assets found for transportation collections.")

files.to_parquet(OUT, index=False)
print("WROTE:", OUT, "rows=", len(files))
print("\nCounts by layer:")
display(files["layer"].value_counts().head(30))

print("\nCounts by storage:")
display(files["storage"].value_counts())

files.head(20)

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_transportation_files.parquet rows= 162

Counts by layer:


layer
segment      122
connector     40
Name: count, dtype: int64


Counts by storage:


storage
aws      81
azure    81
Name: count, dtype: int64

,source_name,release,theme,layer,item_id,asset_key,storage,href,type,xmin,ymin,xmax,ymax,item_url,collection_url
0,overture,2026-02-18.0,transportation,connector,00000,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-180.000000,-85.051128,-5.639153e+00,22.231720,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
1,overture,2026-02-18.0,transportation,connector,00000,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-180.000000,-85.051128,-5.639153e+00,22.231720,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
2,overture,2026-02-18.0,transportation,connector,00001,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-119.571014,16.874998,-9.000000e+01,39.374999,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
3,overture,2026-02-18.0,transportation,connector,00001,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-119.571014,16.874998,-9.000000e+01,39.374999,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
4,overture,2026-02-18.0,transportation,connector,00002,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-180.000000,23.867901,-9.000000e+01,78.796800,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
5,overture,2026-02-18.0,transportation,connector,00002,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-180.000000,23.867901,-9.000000e+01,78.796800,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
6,overture,2026-02-18.0,transportation,connector,00003,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-123.749830,44.999997,-4.000000e-07,82.525402,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
7,overture,2026-02-18.0,transportation,connector,00003,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-123.749830,44.999997,-4.000000e-07,82.525402,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
8,overture,2026-02-18.0,transportation,connector,00004,aws,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,application/vnd.apache.parquet,-78.750007,22.501208,-6.000000e-07,47.384019,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...
9,overture,2026-02-18.0,transportation,connector,00004,azure,azure,https://overturemapswestus2.blob.core.windows....,application/vnd.apache.parquet,-78.750007,22.501208,-6.000000e-07,47.384019,https://stac.overturemaps.org/2026-02-18.0/tra...,https://stac.overturemaps.org/2026-02-18.0/tra...


In [68]:
# ============================================================
# CELL T4 — Transportation: build grid index (NO SAMPLING)
# ============================================================

from pathlib import Path
import duckdb
import pandas as pd
from tqdm.auto import tqdm

CELL_DEG = 0.25  # keep consistent with addresses for now
OUT_PATH = REG / "overture_transportation_grid_index.parquet"

files = pd.read_parquet(REG / "overture_transportation_files.parquet")

# Use AWS only for indexing (Azure mirrors same data)
aws = files[files["storage"].eq("aws")].copy()

print("AWS parts:", len(aws))
print("Layers:", aws["layer"].value_counts().to_dict())

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

def index_one_part(href: str) -> pd.DataFrame:
    # NO SAMPLING: full scan, but only reading bbox + minimal cost
    q = f"""
    WITH s AS (
      SELECT bbox
      FROM read_parquet('{href}')
      WHERE bbox IS NOT NULL
        AND bbox.xmin IS NOT NULL AND bbox.ymin IS NOT NULL
        AND bbox.xmax IS NOT NULL AND bbox.ymax IS NOT NULL
    ),
    b AS (
      SELECT
        CAST(FLOOR((bbox.xmin - (-180.0)) / {CELL_DEG}) AS BIGINT) AS lon0,
        CAST(FLOOR((bbox.xmax - (-180.0)) / {CELL_DEG}) AS BIGINT) AS lon1,
        CAST(FLOOR((bbox.ymin - (-90.0))  / {CELL_DEG}) AS BIGINT) AS lat0,
        CAST(FLOOR((bbox.ymax - (-90.0))  / {CELL_DEG}) AS BIGINT) AS lat1
      FROM s
    )
    SELECT
      lon_bin,
      lat_bin,
      COUNT(*) AS n_in_cell
    FROM (
      SELECT
        UNNEST(GENERATE_SERIES(lon0, lon1)) AS lon_bin,
        UNNEST(GENERATE_SERIES(lat0, lat1)) AS lat_bin
      FROM b
    )
    GROUP BY lon_bin, lat_bin
    """
    return con.execute(q).df()

rows = []
for r in tqdm(aws.itertuples(index=False), total=len(aws), desc="Index parts"):
    df = index_one_part(r.href)
    if df.empty:
        continue
    df["release"] = r.release
    df["theme"] = r.theme
    df["layer"] = r.layer
    df["storage"] = "aws"
    df["href"] = r.href
    df["cell_deg"] = CELL_DEG
    df["cell_id"] = "g0.25_" + df["lon_bin"].astype(str) + "_" + df["lat_bin"].astype(str)
    rows.append(df)

grid = pd.concat(rows, ignore_index=True)
grid = grid[["release","theme","layer","storage","href","cell_id","lon_bin","lat_bin","cell_deg","n_in_cell"]]

grid.to_parquet(OUT_PATH, index=False)
print("WROTE:", OUT_PATH, "rows=", len(grid), "| unique parts:", grid["href"].nunique(), "| unique cells:", grid["cell_id"].nunique())
display(grid.head(20))

AWS parts: 81
Layers: {'segment': 61, 'connector': 20}


Index parts:   0%|          | 0/81 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_transportation_grid_index.parquet rows= 384236 | unique parts: 81 | unique cells: 188759


,release,theme,layer,storage,href,cell_id,lon_bin,lat_bin,cell_deg,n_in_cell
0,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_449_216,449,216,0.25,7
1,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_447_216,447,216,0.25,10
2,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_446_216,446,216,0.25,6
3,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_457_222,457,222,0.25,8
4,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_453_222,453,222,0.25,3
5,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_452_221,452,221,0.25,33
6,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_452_219,452,219,0.25,56
7,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_456_219,456,219,0.25,230
8,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_456_221,456,221,0.25,1
9,2026-02-18.0,transportation,connector,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,g0.25_460_212,460,212,0.25,17


In [71]:
# ============================================================
# CELL T5 — Transportation bbox sample query (4 cities)
# Fix: handles 1+ candidate parts (no duplicate read_parquet alias)
# ============================================================

import duckdb
import pandas as pd

GRID = pd.read_parquet(REG / "overture_transportation_grid_index.parquet")
GRID = GRID[(GRID["storage"] == "aws") & (GRID["layer"] == "connector")].copy()

CELL_DEG = float(GRID["cell_deg"].iloc[0])
print("=" * 80)
print(f"TRANSPORTATION layer = connector | release = {GRID['release'].iloc[0]} | cell_deg = {CELL_DEG}")
print("=" * 80)

# ---- Cities (lon_min, lat_min, lon_max, lat_max) ----
CITIES = {
    "san_diego": (-117.35, 32.60, -116.90, 33.10),
    "berlin":    (13.0884, 52.3383, 13.7611, 52.6755),
    "riyadh":    (46.52, 24.52, 47.02, 24.92),
    "amman":     (35.80, 31.85, 36.10, 32.05),
}

def bbox_to_bins(xmin, ymin, xmax, ymax, cell_deg=CELL_DEG):
    # same bin logic you used elsewhere
    lon_min = int((xmin - (-180.0)) // cell_deg)
    lon_max = int((xmax - (-180.0)) // cell_deg)
    lat_min = int((ymin - (-90.0))  // cell_deg)
    lat_max = int((ymax - (-90.0))  // cell_deg)
    return lon_min, lon_max, lat_min, lat_max

# Columns to return (keep simple + stable across schema differences)
sel = "id, theme, type"

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

for city, (xmin, ymin, xmax, ymax) in CITIES.items():
    print("\n" + "-" * 30)
    print(f"CITY: {city} bbox: {(xmin, ymin, xmax, ymax)}")

    lon_min, lon_max, lat_min, lat_max = bbox_to_bins(xmin, ymin, xmax, ymax)

    # candidate parts from grid index
    cand = GRID[
        (GRID["lon_bin"].between(lon_min, lon_max)) &
        (GRID["lat_bin"].between(lat_min, lat_max))
    ][["href", "lon_bin", "lat_bin"]].drop_duplicates()

    if cand.empty:
        print("Candidate parts: 0 (no grid cells intersect)")
        continue

    hrefs = cand["href"].drop_duplicates().tolist()

    # Quick summary table
    part_summary = (cand.groupby("href")
                    .agg(lon_min=("lon_bin","min"),
                         lon_max=("lon_bin","max"),
                         lat_min=("lat_bin","min"),
                         lat_max=("lat_bin","max"),
                         n_cells=("lon_bin","size"))
                    .reset_index())
    print("Candidate parts:", len(hrefs))
    display(part_summary)

    # ---- FIX: union each parquet as an explicitly-aliased subquery ----
    union_sql = "\nUNION ALL\n".join([
        f"SELECT * FROM read_parquet('{h}') AS t{i}"
        for i, h in enumerate(hrefs)
    ])

    q = f"""
    WITH x AS (
      {union_sql}
    )
    SELECT {sel}
    FROM x
    WHERE bbox IS NOT NULL
      AND bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
      AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
    LIMIT 50
    """

    df = con.execute(q).df()
    print("Rows:", len(df))
    display(df)

TRANSPORTATION layer = connector | release = 2026-02-18.0 | cell_deg = 0.25

------------------------------
CITY: san_diego bbox: (-117.35, 32.6, -116.9, 33.1)
Candidate parts: 1


,href,lon_min,lon_max,lat_min,lat_max,n_cells
0,https://overturemaps-us-west-2.s3.us-west-2.am...,250,252,490,492,9


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 50


,id,theme,type
0,453c9a45-a812-4de2-b55e-de99c9fe0b6f,transportation,connector
1,791fdb4d-143d-4deb-a0dc-6330fa87366a,transportation,connector
2,78e729d7-2365-4d76-a97c-900840c4dc3a,transportation,connector
3,e458cbfa-b2f4-4532-831f-7ebec6d48594,transportation,connector
4,1c0abd71-ff43-4b13-88bf-3851257d3a25,transportation,connector
5,62888852-124a-4dd2-8e1b-6b6d0aab0820,transportation,connector
6,860ae8f8-3d6a-41e0-a758-f6ec248c8786,transportation,connector
7,9eae33f3-b9cb-417a-84ff-f65155fc5309,transportation,connector
8,873d7ea4-b3a9-4b1d-a0fe-532054d6c9ee,transportation,connector
9,e8f544bd-0ab8-4f63-a300-adbb2e3ee072,transportation,connector


...


------------------------------
CITY: amman bbox: (35.8, 31.85, 36.1, 32.05)
Candidate parts: 2


,href,lon_min,lon_max,lat_min,lat_max,n_cells
0,https://overturemaps-us-west-2.s3.us-west-2.am...,863,864,487,488,4
1,https://overturemaps-us-west-2.s3.us-west-2.am...,863,863,488,488,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 50


,id,theme,type
0,64e7c73e-9613-4504-904f-7d1bad597f2c,transportation,connector
1,1cf33221-1e3a-42b2-84b1-fa10cebfbca1,transportation,connector
2,5740408e-3f15-44ff-bd29-740ace22a821,transportation,connector
3,1c61bd35-97d7-4b8b-acc3-f9449f0ff140,transportation,connector
4,bebea44a-673c-4a04-9f14-6e0805f1248b,transportation,connector
5,432d1fc2-b6eb-4fbb-860c-6b31d590a5b1,transportation,connector
6,654bac87-166c-4919-8719-75898fbe895a,transportation,connector
7,98093b8e-bb48-4fef-9cad-ff80e5268f6a,transportation,connector
8,d16fecf5-40f8-4979-a648-8b331bfe21ec,transportation,connector
9,df8a6022-4dd9-46f4-aa4e-14aa5438257f,transportation,connector


In [72]:
# ============================================================
# CELL T4S — Build grid index for transportation/segment (AWS)
# ============================================================

import duckdb
import pandas as pd
from tqdm.auto import tqdm

FILES = pd.read_parquet(REG / "overture_transportation_files.parquet")

aws = FILES[(FILES["storage"] == "aws") & (FILES["layer"] == "segment")].copy()
hrefs = aws["href"].drop_duplicates().tolist()

CELL_DEG = 0.25
OUT = REG / "overture_transportation_segment_grid_index.parquet"

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

pieces = []
for href in tqdm(hrefs, desc="Grid index (segment)"):
    q = f"""
    SELECT
      CAST(FLOOR((bbox.xmin - (-180.0)) / {CELL_DEG}) AS BIGINT) AS lon_bin,
      CAST(FLOOR((bbox.ymin - (-90.0))  / {CELL_DEG}) AS BIGINT) AS lat_bin,
      COUNT(*) AS n_in_cell
    FROM read_parquet('{href}')
    WHERE bbox IS NOT NULL
      AND bbox.xmin IS NOT NULL AND bbox.ymin IS NOT NULL
    GROUP BY lon_bin, lat_bin
    """
    df = con.execute(q).df()
    if df.empty:
        continue
    df["release"] = aws["release"].iloc[0]
    df["theme"] = "transportation"
    df["layer"] = "segment"
    df["storage"] = "aws"
    df["href"] = href
    df["cell_deg"] = CELL_DEG
    df["cell_id"] = "g0.25_" + df["lon_bin"].astype(str) + "_" + df["lat_bin"].astype(str)
    pieces.append(df)

grid = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(
    columns=["release","theme","layer","storage","href","cell_id","lon_bin","lat_bin","cell_deg","n_in_cell"]
)

grid.to_parquet(OUT, index=False)
print(f"WROTE: {OUT} rows= {len(grid):,} | unique parts: {grid['href'].nunique() if len(grid) else 0:,} | unique cells: {grid['cell_id'].nunique() if len(grid) else 0:,}")
grid.head(10)

Grid index (segment):   0%|          | 0/61 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_transportation_segment_grid_index.parquet rows= 181,331 | unique parts: 61 | unique cells: 175,522


,lon_bin,lat_bin,n_in_cell,release,theme,layer,storage,href,cell_deg,cell_id
0,425,181,1,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_425_181
1,430,181,34,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_430_181
2,431,184,18,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_431_184
3,435,185,16,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_435_185
4,436,185,139,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_436_185
5,434,181,14,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_434_181
6,439,181,2,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_439_181
7,440,181,1,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_440_181
8,443,182,14,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_443_182
9,443,180,2,2026-02-18.0,transportation,segment,aws,https://overturemaps-us-west-2.s3.us-west-2.am...,0.25,g0.25_443_180


In [73]:
# ============================================================
# CELL T5 — Transportation bbox sample query (4 cities)
# Fix: handles 1+ candidate parts (no duplicate read_parquet alias)
# ============================================================

import duckdb
import pandas as pd

GRID = pd.read_parquet(REG / "overture_transportation_segment_grid_index.parquet")
GRID = GRID[(GRID["storage"] == "aws") & (GRID["layer"] == "segment")].copy()

CELL_DEG = float(GRID["cell_deg"].iloc[0])
print("=" * 80)
print(f"TRANSPORTATION layer = connector | release = {GRID['release'].iloc[0]} | cell_deg = {CELL_DEG}")
print("=" * 80)

# ---- Cities (lon_min, lat_min, lon_max, lat_max) ----
CITIES = {
    "san_diego": (-117.35, 32.60, -116.90, 33.10),
    "berlin":    (13.0884, 52.3383, 13.7611, 52.6755),
    "riyadh":    (46.52, 24.52, 47.02, 24.92),
    "amman":     (35.80, 31.85, 36.10, 32.05),
}

def bbox_to_bins(xmin, ymin, xmax, ymax, cell_deg=CELL_DEG):
    # same bin logic you used elsewhere
    lon_min = int((xmin - (-180.0)) // cell_deg)
    lon_max = int((xmax - (-180.0)) // cell_deg)
    lat_min = int((ymin - (-90.0))  // cell_deg)
    lat_max = int((ymax - (-90.0))  // cell_deg)
    return lon_min, lon_max, lat_min, lat_max

# Columns to return (keep simple + stable across schema differences)
sel = "id, theme, type"

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

for city, (xmin, ymin, xmax, ymax) in CITIES.items():
    print("\n" + "-" * 30)
    print(f"CITY: {city} bbox: {(xmin, ymin, xmax, ymax)}")

    lon_min, lon_max, lat_min, lat_max = bbox_to_bins(xmin, ymin, xmax, ymax)

    # candidate parts from grid index
    cand = GRID[
        (GRID["lon_bin"].between(lon_min, lon_max)) &
        (GRID["lat_bin"].between(lat_min, lat_max))
    ][["href", "lon_bin", "lat_bin"]].drop_duplicates()

    if cand.empty:
        print("Candidate parts: 0 (no grid cells intersect)")
        continue

    hrefs = cand["href"].drop_duplicates().tolist()

    # Quick summary table
    part_summary = (cand.groupby("href")
                    .agg(lon_min=("lon_bin","min"),
                         lon_max=("lon_bin","max"),
                         lat_min=("lat_bin","min"),
                         lat_max=("lat_bin","max"),
                         n_cells=("lon_bin","size"))
                    .reset_index())
    print("Candidate parts:", len(hrefs))
    display(part_summary)

    # ---- FIX: union each parquet as an explicitly-aliased subquery ----
    union_sql = "\nUNION ALL\n".join([
        f"SELECT * FROM read_parquet('{h}') AS t{i}"
        for i, h in enumerate(hrefs)
    ])

    q = f"""
    WITH x AS (
      {union_sql}
    )
    SELECT {sel}
    FROM x
    WHERE bbox IS NOT NULL
      AND bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
      AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
    LIMIT 50
    """

    df = con.execute(q).df()
    print("Rows:", len(df))
    display(df)

TRANSPORTATION layer = connector | release = 2026-02-18.0 | cell_deg = 0.25

------------------------------
CITY: san_diego bbox: (-117.35, 32.6, -116.9, 33.1)
Candidate parts: 1


,href,lon_min,lon_max,lat_min,lat_max,n_cells
0,https://overturemaps-us-west-2.s3.us-west-2.am...,250,252,490,492,9


Rows: 50


,id,theme,type
0,e12b04cc-b196-47fd-99bc-3639290ea6b2,transportation,segment
1,671c577c-308f-4aab-b7eb-85fcbfa217b5,transportation,segment
2,c53159b8-cfd2-45c9-99e8-3780cc4d6f6f,transportation,segment
3,2daeb4c3-9f35-4517-ac28-8fb3b2343f1b,transportation,segment
4,bc988ad8-01dc-4eef-adec-491112bb48dc,transportation,segment
5,51d35d46-a526-4334-bfe5-796dfc5bbdd3,transportation,segment
6,4f21a3ef-72f3-4fbc-bca5-d101397a90a1,transportation,segment
7,f6313131-dc68-4cdc-8582-14f20fb603b6,transportation,segment
8,4778fb2f-d669-410b-bb73-c88d5959e54d,transportation,segment
9,3a484bfa-4ac2-4c20-83e1-bc36273b27de,transportation,segment



------------------------------
CITY: berlin bbox: (13.0884, 52.3383, 13.7611, 52.6755)
Candidate parts: 1


...


------------------------------
CITY: amman bbox: (35.8, 31.85, 36.1, 32.05)
Candidate parts: 1


,href,lon_min,lon_max,lat_min,lat_max,n_cells
0,https://overturemaps-us-west-2.s3.us-west-2.am...,863,864,487,488,4


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 50


,id,theme,type
0,92bf6fc2-8c0e-4045-9dce-535064083bbe,transportation,segment
1,a25a5a75-112e-45bf-b618-faf0d6a7e3b9,transportation,segment
2,9111f2fb-3be3-4f49-82c6-681caab3cf71,transportation,segment
3,225f091a-129d-42e9-899e-fef5c8d28648,transportation,segment
4,d49a1830-a424-4384-9972-7d2e196d022d,transportation,segment
5,5766b57b-1f7b-4640-b5d8-a296d0f3c5ae,transportation,segment
6,af804b10-3719-4167-8a03-0315582004a0,transportation,segment
7,ad6aa639-2244-43bc-9070-a2dd940afeac,transportation,segment
8,ca690af6-f41f-4dd4-9033-5337a2df5c72,transportation,segment
9,961e565c-30e8-4dd0-8f2f-c22ada607e0c,transportation,segment


In [80]:
# ============================================================
# CELL D1 (FAST) — Overture divisions: collect collections + item URLs (no full crawl)
# ============================================================

from pathlib import Path
import json, hashlib, urllib.parse, urllib.request
import pandas as pd

REG = Path("/Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry")
REG.mkdir(parents=True, exist_ok=True)

RELEASE = "2026-02-18.0"
ROOT = "https://stac.overturemaps.org"
DIVISIONS_CATALOG = f"{ROOT}/{RELEASE}/divisions/catalog.json"
SOURCE_NAME = "overture"
THEME = "divisions"

def stable_id(*parts: str) -> str:
    h = hashlib.blake2b(digest_size=8)
    for p in parts:
        h.update((p or "").encode("utf-8"))
        h.update(b"|")
    return h.hexdigest()

def fetch_json(url: str) -> dict:
    with urllib.request.urlopen(url) as f:
        return json.load(f)

def resolve_href(base_url: str, href: str) -> str:
    return urllib.parse.urljoin(base_url, href)

# 1) Open divisions catalog
div_cat = fetch_json(DIVISIONS_CATALOG)

# 2) Find any collection.json links directly or via 1-level child catalogs
collection_urls = set()
child_catalogs = []

for l in div_cat.get("links", []):
    href = l.get("href")
    if not href:
        continue
    abs_href = resolve_href(DIVISIONS_CATALOG, href)
    if abs_href.endswith("/collection.json"):
        collection_urls.add(abs_href)
    if l.get("rel") == "child" and abs_href.endswith("catalog.json"):
        child_catalogs.append(abs_href)

# 3) One more hop: open child catalogs and pick up their collections
for cat_url in child_catalogs:
    js = fetch_json(cat_url)
    for l in js.get("links", []):
        href = l.get("href")
        if not href:
            continue
        abs_href = resolve_href(cat_url, href)
        if abs_href.endswith("/collection.json"):
            collection_urls.add(abs_href)

print("DIVISIONS catalog:", DIVISIONS_CATALOG)
print("Collections:", len(collection_urls))
if not collection_urls:
    raise RuntimeError("No collection.json found under divisions. Structure changed?")

# 4) For each collection: list item URLs (do NOT open all items)
item_rows = []
file_rows = []

for c_url in sorted(collection_urls):
    c = fetch_json(c_url)
    layer = c.get("id") or None

    item_urls = [resolve_href(c_url, l["href"])
                 for l in c.get("links", [])
                 if l.get("rel") == "item" and l.get("href")]

    for iu in item_urls:
        item_rows.append({
            "source_name": SOURCE_NAME,
            "release": RELEASE,
            "theme": THEME,
            "layer": layer,
            "item_url": iu,
            "collection_url": c_url,
        })

    # Open just the FIRST item to discover parquet assets + storage endpoints
    if item_urls:
        sample_item = fetch_json(item_urls[0])
        assets = sample_item.get("assets") or {}
        for asset_key, a in assets.items():
            href = a.get("href")
            atype = a.get("type")
            if not href:
                continue
            if (isinstance(atype, str) and "parquet" in atype.lower()) or (isinstance(href, str) and ".parquet" in href):
                storage = "aws" if asset_key.lower().startswith("aws") else ("azure" if asset_key.lower().startswith("azure") else asset_key.lower())

                # IMPORTANT: we don't know the full per-item href list here without crawling.
                # So we store the asset_key and we'll obtain actual hrefs in the grid scan stage.
                file_rows.append({
                    "source_name": SOURCE_NAME,
                    "release": RELEASE,
                    "theme": THEME,
                    "layer": layer,
                    "storage": storage,
                    "asset_key": asset_key,
                    "type": atype,
                    "note": "hrefs discovered in grid scan stage (no per-item crawl)",
                })

items_df = pd.DataFrame(item_rows)
files_df = pd.DataFrame(file_rows).drop_duplicates()

items_out = REG / "overture_divisions_items.parquet"
files_out = REG / "overture_divisions_files.parquet"

items_df.to_parquet(items_out, index=False)
files_df.to_parquet(files_out, index=False)

print("WROTE:", items_out, "rows=", len(items_df))
print("WROTE:", files_out, "rows=", len(files_df))
display(items_df.head(10))
display(files_df)

DIVISIONS catalog: https://stac.overturemaps.org/2026-02-18.0/divisions/catalog.json
Collections: 3
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_divisions_items.parquet rows= 6
WROTE: /Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry/overture_divisions_files.parquet rows= 6


,source_name,release,theme,layer,item_url,collection_url
0,overture,2026-02-18.0,divisions,division,https://stac.overturemaps.org/2026-02-18.0/div...,https://stac.overturemaps.org/2026-02-18.0/div...
1,overture,2026-02-18.0,divisions,division_area,https://stac.overturemaps.org/2026-02-18.0/div...,https://stac.overturemaps.org/2026-02-18.0/div...
2,overture,2026-02-18.0,divisions,division_area,https://stac.overturemaps.org/2026-02-18.0/div...,https://stac.overturemaps.org/2026-02-18.0/div...
3,overture,2026-02-18.0,divisions,division_area,https://stac.overturemaps.org/2026-02-18.0/div...,https://stac.overturemaps.org/2026-02-18.0/div...
4,overture,2026-02-18.0,divisions,division_area,https://stac.overturemaps.org/2026-02-18.0/div...,https://stac.overturemaps.org/2026-02-18.0/div...
5,overture,2026-02-18.0,divisions,division_boundary,https://stac.overturemaps.org/2026-02-18.0/div...,https://stac.overturemaps.org/2026-02-18.0/div...


,source_name,release,theme,layer,storage,asset_key,type,note
0,overture,2026-02-18.0,divisions,division,aws,aws,application/vnd.apache.parquet,hrefs discovered in grid scan stage (no per-it...
1,overture,2026-02-18.0,divisions,division,azure,azure,application/vnd.apache.parquet,hrefs discovered in grid scan stage (no per-it...
2,overture,2026-02-18.0,divisions,division_area,aws,aws,application/vnd.apache.parquet,hrefs discovered in grid scan stage (no per-it...
3,overture,2026-02-18.0,divisions,division_area,azure,azure,application/vnd.apache.parquet,hrefs discovered in grid scan stage (no per-it...
4,overture,2026-02-18.0,divisions,division_boundary,aws,aws,application/vnd.apache.parquet,hrefs discovered in grid scan stage (no per-it...
5,overture,2026-02-18.0,divisions,division_boundary,azure,azure,application/vnd.apache.parquet,hrefs discovered in grid scan stage (no per-it...


In [ ]:
# ============================================================
# CELL D2 — Divisions grid index (AWS only) @ 0.25 degrees
# ============================================================

import duckdb
import pandas as pd

CELL_DEG = 0.25
LON0, LAT0 = -180.0, -90.0

files = pd.read_parquet(REG / "overture_divisions_files.parquet")

aws = files[(files["storage"] == "aws") & files["href"].notna()].copy()
hrefs = aws["href"].drop_duplicates().tolist()

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

pieces = []
for href in hrefs:
    q = f"""
    SELECT
      CAST(FLOOR((bbox.xmin - ({LON0})) / {CELL_DEG}) AS BIGINT) AS lon_bin,
      CAST(FLOOR((bbox.ymin - ({LAT0})) / {CELL_DEG}) AS BIGINT) AS lat_bin,
      COUNT(*) AS n_in_cell
    FROM read_parquet('{href}')
    WHERE bbox IS NOT NULL
      AND bbox.xmin IS NOT NULL AND bbox.ymin IS NOT NULL
    GROUP BY lon_bin, lat_bin
    """
    df = con.execute(q).df()
    if df.empty:
        continue
    df["release"] = RELEASE
    df["theme"] = THEME
    df["layer"] = aws["layer"].iloc[0] if aws["layer"].nunique() == 1 else "mixed"
    df["storage"] = "aws"
    df["href"] = href
    df["cell_deg"] = CELL_DEG
    df["cell_id"] = df.apply(lambda r: f"g{CELL_DEG}_{int(r.lon_bin)}_{int(r.lat_bin)}", axis=1)

    # optional: cell bbox (deterministic from bins)
    df["cell_xmin"] = LON0 + df["lon_bin"] * CELL_DEG
    df["cell_ymin"] = LAT0 + df["lat_bin"] * CELL_DEG
    df["cell_xmax"] = df["cell_xmin"] + CELL_DEG
    df["cell_ymax"] = df["cell_ymin"] + CELL_DEG

    pieces.append(df)

grid = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=[
    "release","theme","layer","storage","href",
    "cell_id","lon_bin","lat_bin","cell_deg","n_in_cell",
    "cell_xmin","cell_ymin","cell_xmax","cell_ymax"
])

out_path = REG / "overture_divisions_grid_index.parquet"
grid.to_parquet(out_path, index=False)

print("WROTE:", out_path, "rows=", len(grid),
      "| unique parts:", grid["href"].nunique() if len(grid) else 0,
      "| unique cells:", grid["cell_id"].nunique() if len(grid) else 0)
display(grid.head(20))

In [1]:
# ============================================================
# QUICK LIVE DEMO — Overture Places for PB or Rancho Bernardo
# Uses staged file inventory built in your registry notebook
# ============================================================

from pathlib import Path
import time
import pandas as pd
import duckdb
import folium

# --- adjust if needed ---
REG = Path("/Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry")

AREAS = {
    "pb": {
        "label": "Pacific Beach",
        # approximate PB bbox
        "bbox": (-117.257, 32.785, -117.226, 32.809)
    },
    "rancho_bernardo": {
        "label": "Rancho Bernardo",
        # approximate Rancho Bernardo bbox
        "bbox": (-117.130, 33.005, -117.055, 33.050)
    }
}

def run_places_demo(area_key="pb", limit=300):
    area = AREAS[area_key]
    minx, miny, maxx, maxy = area["bbox"]

    t0 = time.perf_counter()

    # 1) staged inventory lookup
    files = pd.read_parquet(REG / "overture_places_files.parquet")
    files = files[files["storage"] == "aws"].copy()

    cand = files[
        (files["xmax"] >= minx) &
        (files["xmin"] <= maxx) &
        (files["ymax"] >= miny) &
        (files["ymin"] <= maxy)
    ].copy()

    t1 = time.perf_counter()

    print(f"Area: {area['label']}")
    print(f"Candidate parquet parts from staged index: {len(cand)}")
    if cand.empty:
        raise RuntimeError("No candidate parts found. Expand bbox or verify staged places index.")

    urls = cand["href"].drop_duplicates().tolist()

    # 2) remote query only candidate parts
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("PRAGMA enable_progress_bar=false;")
    con.execute("PRAGMA threads=4;")

    q = """
    SELECT
        id,
        names.primary AS name,
        categories.primary AS category,
        confidence,
        ST_Y(ST_Centroid(geometry)) AS lat,
        ST_X(ST_Centroid(geometry)) AS lon
    FROM read_parquet($1)
    WHERE ST_Intersects(
        geometry,
        ST_MakeEnvelope($2, $3, $4, $5)
    )
    LIMIT $6
    """

    df = con.execute(q, [urls, minx, miny, maxx, maxy, limit]).df()
    t2 = time.perf_counter()

    print(f"Rows returned: {len(df)}")
    print(f"Index filter time: {t1 - t0:,.2f}s")
    print(f"Remote query time: {t2 - t1:,.2f}s")
    print(f"Total time: {t2 - t0:,.2f}s")

    # 3) quick map
    center_lat = (miny + maxy) / 2
    center_lon = (minx + maxx) / 2
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="CartoDB positron")

    # bbox outline
    folium.Rectangle(
        bounds=[[miny, minx], [maxy, maxx]],
        fill=False,
        weight=2
    ).add_to(m)

    for _, r in df.dropna(subset=["lat", "lon"]).iterrows():
        popup = f"""
        <b>{r['name'] if pd.notna(r['name']) else 'Unnamed place'}</b><br>
        Category: {r['category'] if pd.notna(r['category']) else 'NA'}<br>
        Confidence: {r['confidence'] if pd.notna(r['confidence']) else 'NA'}<br>
        ID: {r['id']}
        """
        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=3,
            popup=folium.Popup(popup, max_width=300),
            fill=True
        ).add_to(m)

    return df, m

In [2]:
df_pb, map_pb = run_places_demo("pb", limit=300)
map_pb

Area: Pacific Beach
Candidate parquet parts from staged index: 1
Rows returned: 300
Index filter time: 0.07s
Remote query time: 44.25s
Total time: 44.32s


In [3]:
df_rb, map_rb = run_places_demo("rancho_bernardo", limit=300)
map_rb

Area: Rancho Bernardo
Candidate parquet parts from staged index: 1
Rows returned: 300
Index filter time: 0.01s
Remote query time: 41.35s
Total time: 41.36s


In [4]:
from pathlib import Path
import time
import pandas as pd
import duckdb
import folium

REG = Path("/Users/mohamadyassin/Documents/AIRC/home_page/open-data/registry")

BBOXES = {
    "pb": (-117.257, 32.785, -117.226, 32.809),
    "rancho_bernardo": (-117.130, 33.005, -117.055, 33.050),
}

def query_places_indexed(area_key="pb", limit=300):
    minx, miny, maxx, maxy = BBOXES[area_key]

    t0 = time.perf_counter()

    files = pd.read_parquet(REG / "overture_places_files.parquet")
    files = files[files["storage"] == "aws"].copy()

    cand = files[
        (files["xmax"] >= minx) &
        (files["xmin"] <= maxx) &
        (files["ymax"] >= miny) &
        (files["ymin"] <= maxy)
    ].copy()

    urls = cand["href"].drop_duplicates().tolist()
    t1 = time.perf_counter()

    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("PRAGMA threads=4;")

    q = """
    SELECT
        id,
        names.primary AS name,
        categories.primary AS category,
        confidence,
        ST_Y(ST_Centroid(geometry)) AS lat,
        ST_X(ST_Centroid(geometry)) AS lon
    FROM read_parquet($1)
    WHERE ST_Intersects(
        geometry,
        ST_MakeEnvelope($2, $3, $4, $5)
    )
    LIMIT $6
    """

    df = con.execute(q, [urls, minx, miny, maxx, maxy, limit]).df()
    t2 = time.perf_counter()

    return {
        "df": df,
        "candidate_parts": len(urls),
        "index_time": t1 - t0,
        "query_time": t2 - t1,
        "total_time": t2 - t0,
    }

In [5]:
def query_places_naive(area_key="pb", limit=300, max_parts=None):
    minx, miny, maxx, maxy = BBOXES[area_key]

    t0 = time.perf_counter()

    files = pd.read_parquet(REG / "overture_places_files.parquet")
    files = files[files["storage"] == "aws"].copy()

    urls = files["href"].drop_duplicates().tolist()

    # optional safety cap for live demos
    if max_parts is not None:
        urls = urls[:max_parts]

    t1 = time.perf_counter()

    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("PRAGMA threads=4;")

    q = """
    SELECT
        id,
        names.primary AS name,
        categories.primary AS category,
        confidence,
        ST_Y(ST_Centroid(geometry)) AS lat,
        ST_X(ST_Centroid(geometry)) AS lon
    FROM read_parquet($1)
    WHERE ST_Intersects(
        geometry,
        ST_MakeEnvelope($2, $3, $4, $5)
    )
    LIMIT $6
    """

    df = con.execute(q, [urls, minx, miny, maxx, maxy, limit]).df()
    t2 = time.perf_counter()

    return {
        "df": df,
        "candidate_parts": len(urls),
        "prep_time": t1 - t0,
        "query_time": t2 - t1,
        "total_time": t2 - t0,
    }

In [6]:
def compare_indexed_vs_naive(area_key="pb", limit=300, naive_max_parts=None):
    fast = query_places_indexed(area_key=area_key, limit=limit)
    slow = query_places_naive(area_key=area_key, limit=limit, max_parts=naive_max_parts)

    print(f"Area: {area_key}")
    print("-" * 60)
    print("INDEXED")
    print(f"  candidate parts : {fast['candidate_parts']}")
    print(f"  index time      : {fast['index_time']:.2f}s")
    print(f"  query time      : {fast['query_time']:.2f}s")
    print(f"  total time      : {fast['total_time']:.2f}s")
    print(f"  rows returned   : {len(fast['df'])}")
    print()
    print("NAIVE REMOTE")
    print(f"  candidate parts : {slow['candidate_parts']}")
    print(f"  prep time       : {slow['prep_time']:.2f}s")
    print(f"  query time      : {slow['query_time']:.2f}s")
    print(f"  total time      : {slow['total_time']:.2f}s")
    print(f"  rows returned   : {len(slow['df'])}")

    if fast["total_time"] > 0:
        print()
        print(f"Speedup: {slow['total_time'] / fast['total_time']:.1f}x")

    return fast, slow

In [7]:
fast_pb, slow_pb = compare_indexed_vs_naive(
    area_key="pb",
    limit=300,
    naive_max_parts=200  # optional cap for a safer live demo
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Area: pb
------------------------------------------------------------
INDEXED
  candidate parts : 1
  index time      : 0.01s
  query time      : 43.13s
  total time      : 43.14s
  rows returned   : 300

NAIVE REMOTE
  candidate parts : 8
  prep time       : 0.00s
  query time      : 359.20s
  total time      : 359.21s
  rows returned   : 300

Speedup: 8.3x


In [8]:
def make_map(df, area_key="pb"):
    minx, miny, maxx, maxy = BBOXES[area_key]
    center_lat = (miny + maxy) / 2
    center_lon = (minx + maxx) / 2

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="CartoDB positron")

    folium.Rectangle(
        bounds=[[miny, minx], [maxy, maxx]],
        fill=False,
        weight=2
    ).add_to(m)

    for _, r in df.dropna(subset=["lat", "lon"]).iterrows():
        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=3,
            popup=f"{r.get('name', 'Unnamed')} | {r.get('category', 'NA')}",
            fill=True
        ).add_to(m)

    return m

In [9]:
make_map(fast_pb["df"], "pb")
make_map(slow_pb["df"], "pb")